# Scan 3D mộc bản → ảnh 2D (render giả ảnh chụp + tăng cường hình học)

Từ **model 3D scan thật** của khối mộc bản / khối in, sinh ảnh 2D để làm giàu dữ liệu huấn luyện OCR. Notebook chạy
cho **mọi scan** trong dataset đầu vào, theo hai hướng:

| Hướng | Làm gì | Cần |
|---|---|---|
| **A — render giả ảnh chụp** | rasterize bằng pyrender với texture, nhiều **góc chụp** + **điều kiện ánh sáng**, hậu kỳ giả camera (phơi sáng, WB, nhiễu, mờ, JPEG), tuỳ chọn vật che 2D | **GPU** (EGL) |
| **B — tăng cường hình học** | chiếu scan thành **ảnh độ sâu** ở góc nhìn bất kỳ rồi làm nổi nét khắc bằng 6 phương pháp (độ sâu cục bộ, pháp tuyến, độ cong, MSII, AO, exaggerated shading) — **không dùng đèn**, nên không phụ thuộc góc chiếu sáng | CPU |

```
scan ──> nạp 1 lần + soát mặt khắc (chung) ──┬─> A: sample_shot → pyrender → hậu kỳ camera → vật che → JPG + meta.json
                                             └─> B: project_depth → ảnh độ sâu → 6 phép tăng cường → PNG + batch_meta.json
```

**Mục lục**
1. Cài đặt
2. **Thư viện hàm** — mỗi hàm một cell, có giải thích chi tiết ngay trước: phần nạp scan & render, phần tăng cường
3. Cấu hình (mọi tham số ở một chỗ) + kiểm tra GPU
4. Dò scan → nạp + soát mặt khắc (mỗi scan nạp **một lần**, dùng lại cho A và B)
5. Phần A — render giả ảnh chụp
6. Phần B — tăng cường hình học
7. Đóng gói `outputs.zip`

Notebook không ghi file mã nguồn nào ra thư mục output: mọi hàm được định nghĩa thẳng trong notebook. Output chỉ gồm
ảnh, meta và `outputs.zip`.

**Settings:** Accelerator = **GPU** (T4/P100), Internet = **On** (để pip cài).
Không bật GPU thì phần A **tự bỏ qua**, phần B vẫn chạy đủ.

**Dữ liệu:** notebook chạy trên mọi scan có trong dataset đầu vào. Có scan mộc bản chữ Hán thì chỉ cần đổi dataset.

## 0. Chuẩn bị dữ liệu: tải lên Kaggle Dataset
Kaggle **không** tự tải được scan từ trang nguồn (thường yêu cầu đăng nhập). Làm 1 lần:

1. Tải model về máy (định dạng gốc OBJ hoặc glTF/GLB) → file `.zip`.
2. Kaggle → **Create → New Dataset** → kéo các file `.zip` vào (Kaggle tự giải nén, kể cả zip lồng bên trong). Một
   dataset chứa được **nhiều scan**; **tên file mesh không được trùng nhau**.
3. Notebook này: panel phải **Add Input → Datasets → Your Datasets** → chọn dataset.
4. *(Tuỳ chọn)* thêm `manifest.csv` vào dataset để chỉ định tay mặt khắc (xem mục soát bên dưới).

Texture: GLB thường nhúng sẵn. OBJ scan thường có ảnh rời (vd. `textures/xxx_4K.jpg`) — notebook tự tìm ảnh có tên
trùng tiền tố với mesh ở thư mục mesh và các thư mục xung quanh (lên tới 3 cấp), bỏ qua normal map / roughness.
Texture chỉ dùng cho hướng A.

## 1. Cài đặt
Chỉ cài gói **còn thiếu** (Kaggle có sẵn numpy / scipy / opencv / matplotlib), in tiến trình pip ngay khi chạy, và kiểm tra Internet trước để không treo im lặng.

In [ ]:
import os, sys, subprocess, platform, socket, importlib.util, importlib.metadata as imd
# EGL headless phải đặt TRƯỚC mọi lần import OpenGL/pyrender: PyOpenGL chọn nền tảng ngay lúc import lần đầu
os.environ["PYOPENGL_PLATFORM"] = "egl"
print(platform.platform(), sys.version)
!nvidia-smi -L || echo "Không có GPU -> phần A (pyrender/EGL) sẽ bị bỏ qua, phần B vẫn chạy"

def have(mod):
    return importlib.util.find_spec(mod) is not None

def dist_version(name):
    try:
        return imd.version(name)
    except imd.PackageNotFoundError:
        return None

def pip(*args):
    # KHÔNG capture: in từng dòng ngay khi pip chạy để thấy đang tải gì; timeout mạng ngắn để không treo im lặng
    cmd = [sys.executable, "-m", "pip", "install", "--progress-bar", "off", "--timeout", "30", "--retries", "2", *args]
    print("$ pip install", " ".join(args), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if not line.startswith("Requirement already satisfied"):
            print("   ", line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"pip lỗi: {args}")

# Kaggle có sẵn numpy / scipy / opencv / matplotlib / pillow: KHÔNG cài lại (cài opencv-python-headless mới nhất
# sẽ kéo nâng cấp numpy -> rất lâu và dễ vỡ môi trường). Chỉ cài phần còn thiếu.
need = [pkg for mod, pkg in [("trimesh", "trimesh"), ("pyglet", "pyglet"), ("freetype", "freetype-py"),
                             ("imageio", "imageio"), ("six", "six"), ("networkx", "networkx"),
                             ("cv2", "opencv-python-headless"), ("scipy", "scipy"), ("matplotlib", "matplotlib"),
                             ("PIL", "pillow")] if not have(mod)]
# pyrender 0.1.45 pin PyOpenGL==3.1.0 (lỗi glGenTextures trên Python mới) -> cài no-deps + PyOpenGL 3.1.7
if dist_version("PyOpenGL") != "3.1.7":
    need.append("PyOpenGL==3.1.7")
need_pyrender = not have("pyrender")
print("cần cài:", need + (["pyrender (--no-deps)"] if need_pyrender else []) or "không — đủ cả")

if need or need_pyrender:
    try:
        socket.create_connection(("pypi.org", 443), timeout=5).close()
    except OSError:
        raise SystemExit("Không kết nối được pypi.org: bật Settings → Internet = On rồi chạy lại cell này")
    if need:
        pip(*need)
    if need_pyrender:
        pip("--no-deps", "pyrender")

import trimesh, cv2, numpy, scipy
print("trimesh", trimesh.__version__, "| cv2", cv2.__version__, "| numpy", numpy.__version__, "| scipy", scipy.__version__,
      "| PyOpenGL", dist_version("PyOpenGL"), "| pyrender", dist_version("pyrender"))

## 2. Thư viện hàm
Mỗi hàm là một cell, đứng ngay sau phần giải thích: **làm gì**, **đầu vào / đầu ra**, **cách làm** (công thức) và
**lưu ý** (những lỗi đã gặp khi chạy trên scan thật). Chạy tuần tự mọi cell trong mục này trước khi sang mục 3.
Mã nguồn giống hệt hai package `src/core/mocban_render/` và `src/core/mocban_enhance/` trong repo (mỗi mục = một file).

## 2A. Nạp scan & render (`mocban_render/`)
Dùng chung cho soát mặt khắc, phần A và phần B.

#### Import & shim
Thư viện dùng cho phần nạp scan và render. `trimesh` đọc mesh, `cv2` / `numpy` xử lý ảnh, `PIL` đọc texture.

**Shim `np.infty`:** pyrender 0.1.45 (bản cuối trên PyPI, 2020) còn gọi `np.infty`, tên này bị bỏ ở NumPy 2. Gán lại
`np.infty = np.inf` **trước** khi import pyrender, nếu không pyrender lỗi ngay khi import.

In [ ]:
from dataclasses import dataclass, field, asdict
import json
import math
import os
from pathlib import Path
import random
from typing import Optional
from PIL import Image
import cv2
import numpy as np
import trimesh
# pyrender 0.1.45 (PyPI, 2020) còn dùng np.infty đã bị bỏ ở NumPy 2 -> shim trước khi import pyrender
if not hasattr(np, "infty"):
    np.infty = np.inf

### 1. Nạp mesh scan + chuẩn hoá khung (OBB)
Nhận file scan bất kỳ, tìm texture rời, đưa scan về một khung toạ độ chuẩn (OBB, quay thật, scale về `TARGET_MM`). File `s01_scan.py`.

#### Hằng số
Hằng số của bước nạp scan:

| Hằng số | Giá trị | Ý nghĩa |
|---|---|---|
| `MESH_EXT` | glb, gltf, obj, stl, ply | Đuôi file được coi là mesh |
| `_IMG_EXT` | jpg, jpeg, png | Đuôi file được coi là ảnh texture |
| `_NOT_ALBEDO` | `_nm`, `normal`, `_rough`… | Tên ảnh chứa các chữ này là **không phải** ảnh màu (normal map, roughness…) nên bị bỏ |

In [ ]:
MESH_EXT = (".glb", ".gltf", ".obj", ".stl", ".ply")
_IMG_EXT = (".jpg", ".jpeg", ".png")
_NOT_ALBEDO = ("_nm", "normal", "_disp", "_height", "_rough", "_metal", "_ao", "_spec", "_mask", "_bump")

#### `_has_texture_image(m)`
**Làm gì:** kiểm tra mesh có ảnh texture màu **thật** hay không.

**Cách làm:** lấy `mesh.visual.material.image` (OBJ) hoặc `baseColorTexture` (GLB); có ảnh và cạnh nhỏ nhất ≥ 16 px
thì trả `True`.

**Vì sao cần ngưỡng 16 px:** với OBJ không kèm file `.mtl`, trimesh tự gắn một ảnh giữ chỗ 2 × 2 px. Không loại ảnh này
thì hàm tưởng mesh đã có texture và bỏ qua bước tự tìm ảnh rời → render ra màu xám.

In [ ]:
def _has_texture_image(m) -> bool:
    """Mesh có ảnh texture thật? (trimesh gắn ảnh giữ chỗ 2x2 cho OBJ không kèm .mtl -> không tính)."""
    mat = getattr(m.visual, "material", None)
    img = getattr(mat, "image", None) or getattr(mat, "baseColorTexture", None)
    return img is not None and min(img.size) >= 16

#### `_texture_search_dirs(mesh_path, up)`
**Làm gì:** liệt kê các thư mục cần tìm ảnh texture cho một file mesh.

**Đầu vào:** `mesh_path`; `up` = số cấp thư mục cha được xét (mặc định 1).

**Đầu ra:** danh sách thư mục, không trùng, theo thứ tự gần → xa:
1. thư mục chứa mesh;
2. với mỗi cấp cha (tới `up` cấp): chính thư mục cha đó và **các thư mục con trực tiếp** của nó.

**Ví dụ:** `up=1` tìm được `<scan>/textures/` khi mesh ở `<scan>/source/x.obj`. `up=2` cần khi Kaggle giải nén zip lồng
thành `<scan>/source/<zip_long>/x.obj` — mesh sâu thêm một cấp so với thư mục `textures/`.

In [ ]:
def _texture_search_dirs(mesh_path: Path, up: int = 1) -> list[Path]:
    """Thư mục của mesh + các thư mục con trực tiếp của `up` cấp tổ tiên (up=1: thư mục anh em, vd.
    <scan>/source/x.obj cạnh <scan>/textures/x_4K.jpg; up=2: thêm một cấp, vd. Kaggle giải nén zip lồng thành
    <scan>/source/<zip_long>/x.obj trong khi texture ở <scan>/textures/)."""
    dirs = [mesh_path.parent]
    anc = mesh_path.parent
    for _ in range(up):
        if anc.parent == anc:
            break
        anc = anc.parent
        dirs += [anc] + [d for d in anc.iterdir() if d.is_dir() and d not in dirs]
    return list(dict.fromkeys(dirs))

#### `find_texture(mesh_path)`
**Làm gì:** tự tìm ảnh texture màu cho mesh có UV nhưng không kèm ảnh.

**Đầu vào:** đường dẫn mesh. **Đầu ra:** đường dẫn ảnh, hoặc `None` nếu không chắc chắn.

**Cách làm:**
1. Ứng viên = mọi file `.jpg/.jpeg/.png` trong các thư mục của `_texture_search_dirs`, bỏ ảnh có tên kiểu normal map /
   roughness / AO… (`_NOT_ALBEDO`).
2. Điểm mỗi ứng viên = **độ dài tiền tố chung** giữa tên ảnh và tên mesh (không phân biệt hoa thường); hoà thì chọn
   file lớn hơn (thường là bản độ phân giải cao).
3. Hai vòng tìm:
   - **gần** (thư mục mesh + anh em): nhận nếu tiền tố chung ≥ 4 ký tự, **hoặc** chỉ có đúng 1 ứng viên;
   - **xa** (lên tới 3 cấp cha): **bắt buộc** tiền tố chung ≥ 4 ký tự, vì càng lên cao càng dễ gặp texture của scan
     khác trong cùng dataset.

**Không đoán bừa:** nhiều ảnh mà không ảnh nào trùng tên → trả `None`; khi đó ghi tên ảnh vào cột `texture` của
manifest.

Ví dụ: mesh `block.obj` và ảnh `block_albedo.jpg` có tiền tố chung dài → chọn đúng, bỏ qua `block_normal.png`.

In [ ]:
def find_texture(mesh_path: str | Path) -> Optional[Path]:
    """Ảnh texture màu cho mesh có UV nhưng không kèm material. Bỏ normal/roughness/..., chọn ảnh có tên chung
    tiền tố dài nhất với tên mesh (hoà -> file lớn nhất). None nếu không có ảnh, hoặc có nhiều ảnh mà không ảnh
    nào trùng tên (không đoán bừa: khi đó chỉ định trong manifest).
    Tìm gần trước (thư mục mesh + anh em); không thấy thì lên thêm 2 cấp nhưng khi đó BẮT BUỘC trùng tiền tố tên,
    vì càng lên cao càng dễ gặp texture của scan khác."""
    p = Path(mesh_path)
    stem = p.stem.lower()

    def pref(f):
        return len(os.path.commonprefix([stem, f.stem.lower()]))
    for up, strict in ((1, False), (3, True)):
        cands = list(dict.fromkeys(
            f for d in _texture_search_dirs(p, up) for f in d.iterdir()
            if f.is_file() and f.suffix.lower() in _IMG_EXT and not any(k in f.stem.lower() for k in _NOT_ALBEDO)))
        if not cands:
            continue
        best = max(cands, key=lambda f: (pref(f), f.stat().st_size))
        if pref(best) >= 4 or (len(cands) == 1 and not strict):
            return best
    return None

#### `resolve_texture(mesh_path, texture)`
**Làm gì:** quyết định texture cuối cùng cho một mesh từ giá trị người dùng đưa vào.

| `texture` | Kết quả |
|---|---|
| `None` | không gắn texture (phần B dùng, vì chỉ cần hình học) |
| `""` hoặc `"auto"` | gọi `find_texture` |
| đường dẫn tồn tại | dùng luôn |
| tên file / đường dẫn không tồn tại | tìm file **cùng tên** trong các thư mục quanh mesh (tới 3 cấp cha) |

Tìm theo tên để manifest viết ở máy này (đường dẫn Windows) vẫn dùng được trên Kaggle. Không thấy → báo lỗi
`FileNotFoundError` thay vì lặng lẽ render xám.

In [ ]:
def resolve_texture(mesh_path: str | Path, texture: str | Path | None) -> Optional[Path]:
    """None -> không gắn texture. 'auto' / '' -> find_texture. Tên file hoặc đường dẫn không tồn tại -> tìm
    theo TÊN trong các thư mục cạnh mesh (để manifest viết ở máy này vẫn dùng được trên Kaggle)."""
    if texture is None:
        return None
    if texture in ("", "auto"):
        return find_texture(mesh_path)
    t = Path(texture)
    if t.exists():
        return t
    for d in _texture_search_dirs(Path(mesh_path), up=3):
        if (d / t.name).exists():
            return d / t.name
    raise FileNotFoundError(f"không thấy texture {texture} cho {mesh_path}")

#### `_proper_rotation(R)`
**Làm gì:** biến ma trận 3 × 3 thành phép **quay thật** (định thức = +1).

**Cách làm:** nếu `det(R) < 0` (ma trận có chứa phép lật gương) thì đổi dấu hàng thứ 3.

**Vì sao:** trục OBB mà trimesh trả về có thể là hệ tay trái. Áp nguyên ma trận đó sẽ **lật gương** mesh, và chữ khắc
trên mộc bản sẽ bị đảo chiều — lỗi rất khó nhận ra bằng mắt.

In [ ]:
def _proper_rotation(R: np.ndarray) -> np.ndarray:
    """Ép ma trận quay 3x3 về det = +1 (đảo dấu một trục) -> không lật gương."""
    R = np.array(R, dtype=float)
    if np.linalg.det(R) < 0:
        R[2] *= -1
    return R

#### `prepare_scan(src, target_extent_mm, texture_path)`
**Làm gì:** nạp scan và đưa về **khung OBB chuẩn** — khung chung để chấm điểm các mặt, không phụ thuộc scan được đặt thế
nào trong file.

**Đầu vào:** `src` (đường dẫn hoặc `trimesh.Trimesh`), `target_extent_mm` (cạnh dài nhất sau khi scale, mặc định 200),
`texture_path` (như `resolve_texture`).

**Đầu ra:** mesh mới: tâm OBB ở gốc toạ độ, 3 cạnh OBB song song 3 trục, cạnh dài nhất = `target_extent_mm`.

**Các bước:**
1. `trimesh.load(..., process=False)`: **không** dùng `force="mesh"` vì tuỳ chọn đó làm mất texture/UV. Scene nhiều node
   (GLB) được gộp bằng `to_mesh()`, có áp phép biến đổi của từng node.
2. Mesh có UV mà chưa có ảnh → tìm và gắn ảnh rời bằng `TextureVisuals(uv, image)`.
3. **Sửa mesh lộn trong ra ngoài:** tính `s = Σ (area · (n·r)/|r|) / tổng diện tích`, với `r` là vector từ tâm tới tâm
   tam giác, `n` là pháp tuyến. Mesh đúng thì pháp tuyến hướng ra ngoài (`s > 0`); `s < −0,2` → lật mọi mặt. Cần vì
   bước chấm điểm dựa vào hướng pháp tuyến.
4. Lấy OBB, ép quay thật (`_proper_rotation`), dời tâm OBB về gốc: `M = [R | −R·t]`.
5. Scale đều để cạnh dài nhất = `target_extent_mm`.

**Metadata ghi lại:** `to_obb_frame` (ma trận 4 × 4 từ toạ độ file sang khung OBB, để truy ngược), `source_texture`
(ảnh rời đã gắn hoặc `None`), `has_texture` (có ảnh màu thật không; `False` → render phần A sẽ xám).

In [ ]:
def prepare_scan(src, target_extent_mm: float = 200.0, texture_path: str | Path | None = "auto") -> trimesh.Trimesh:
    """Nạp scan (đường dẫn hoặc Trimesh) và đưa về KHUNG OBB: tâm OBB ở gốc, cạnh OBB song song trục toạ độ,
    cạnh dài nhất = target_extent_mm. Khung chung để chấm điểm các mặt, không phụ thuộc scan đặt nghiêng.
    Giữ texture (UV); texture_path='auto' tự tìm ảnh rời khi mesh có UV nhưng không kèm material, None = không gắn.
    metadata['to_obb_frame'] = ma trận 4x4 từ toạ độ file -> khung OBB (để truy lại)."""
    if isinstance(src, trimesh.Trimesh):
        m = src.copy()
    else:
        m = trimesh.load(str(src), process=False)   # KHÔNG force="mesh": giữ TextureVisuals/UV
        if isinstance(m, trimesh.Scene):
            m = m.to_mesh()                          # áp phép biến đổi của từng node rồi gộp
        uv = getattr(m.visual, "uv", None)
        tex = resolve_texture(src, texture_path) if (uv is not None and not _has_texture_image(m)) else None
        if tex is not None:
            m.visual = trimesh.visual.TextureVisuals(uv=uv, image=Image.open(str(tex)).convert("RGB"))
        m.metadata["source_texture"] = str(tex) if tex is not None else None
    # có ảnh màu thật (nhúng sẵn hoặc vừa gắn)? False -> render hướng A sẽ ra màu xám
    m.metadata["has_texture"] = _has_texture_image(m)
    # mesh lộn trong ra ngoài (normal hướng vào trong) -> lật lại, vì chấm điểm dựa vào hướng normal
    o = m.bounding_box.centroid
    rel = m.triangles_center - o
    w = m.area_faces / np.maximum(np.linalg.norm(rel, axis=1), 1e-9)
    if (w * (m.face_normals * rel).sum(1)).sum() / max(m.area, 1e-9) < -0.2:
        m.invert()
    box = m.bounding_box_oriented.primitive.transform     # khung OBB -> toạ độ file
    R = _proper_rotation(box[:3, :3].T)
    M = np.eye(4); M[:3, :3] = R; M[:3, 3] = -R @ box[:3, 3]
    m.apply_transform(M)
    s = target_extent_mm / m.bounding_box.extents.max()
    m.apply_scale(s)
    m.metadata["to_obb_frame"] = (np.diag([s, s, s, 1.0]) @ M).tolist()
    return m

### 2. Tự xác định mặt khắc + manifest
Tìm mặt khắc → xoay mặt khắc hướng lên camera; đọc `manifest.csv` để chỉ định tay. File `s02_face.py`.

Giả định: mỗi khối chỉ có **một** mặt mang thông tin. Để dùng được cho mọi bộ scan:
- hướng tính theo **hộp bao có hướng (OBB)**, nên scan đặt nghiêng thế nào trong file cũng được;
- mọi ngưỡng tính theo **tỉ lệ cạnh dài** của khối, nên không phụ thuộc đơn vị file (mm, m…);
- quyết định bằng **so sánh tương đối** giữa các mặt, kèm độ tin cậy; không chắc thì đánh dấu để người soát;
- chỉ dùng **phép quay thật** (det = +1), nên mesh không bao giờ bị lật gương (chữ khắc sẽ bị đảo);
- luôn có đường chỉ định tay (tham số `face` hoặc file `manifest.csv`).

**Tên mặt:** 3 trục OBB xếp theo độ dài giảm dần `A ≥ B ≥ C` (C = chiều dày). `C+` / `C-` là hai mặt vuông góc trục C. Tên cố định với cùng một file scan, nên dùng được trong manifest.

#### Hằng số
Ngưỡng của bước tìm mặt khắc:

| Hằng số | Giá trị | Ý nghĩa |
|---|---|---|
| `FACE_THIN_RATIO` | 0,6 | `C/B ≤ 0,6` → khối dẹt, chỉ so 2 mặt lớn `C+`, `C-`; ngược lại so cả 6 mặt |
| `FACE_MIN_RATIO` | 2,0 | Mặt tốt nhất phải chi tiết gấp ≥ 2 lần mặt thứ nhì mới **tin cậy** |
| `FACE_MIN_COVERAGE` | 0,5 | Mặt phải có dữ liệu scan trên ≥ 50 % diện tích mới được xếp hạng |
| `FACE_MISSING` | 0,15 | Mặt đối diện có dữ liệu < 15 % → scan chỉ quét một mặt → tin cậy luôn |

Các ngưỡng đặt thử trên 1 scan thật, cần hiệu chỉnh khi có thêm scan.

In [ ]:
FACE_THIN_RATIO = 0.6    # cạnh ngắn / cạnh giữa <= ngưỡng -> khối dẹt: chỉ xét 2 mặt lớn C+/C-, ngược lại xét cả 6
FACE_MIN_RATIO = 2.0     # điểm chi tiết mặt tốt nhất / mặt thứ nhì >= ngưỡng -> tin cậy (hiệu chỉnh khi có thêm scan)
FACE_MIN_COVERAGE = 0.5  # mặt được chọn phải có dữ liệu scan trên >= 50% diện tích
FACE_MISSING = 0.15      # mặt đối diện có dữ liệu < 15% diện tích -> scan chỉ quét một mặt

#### `_axis_names(ext)`
**Làm gì:** đặt tên `A`, `B`, `C` cho 3 trục theo độ dài cạnh giảm dần.

**Đầu vào:** `ext` = 3 kích thước hộp bao. **Đầu ra:** dict `{chỉ số trục: tên}`, ví dụ `{0: 'B', 1: 'A', 2: 'C'}`.

Sắp xếp ổn định (`kind="stable"`): hai cạnh dài bằng nhau thì trục có chỉ số nhỏ hơn đứng trước, nên tên không nhảy
giữa các lần chạy.

In [ ]:
def _axis_names(ext) -> dict[int, str]:
    return {int(ax): "ABC"[i] for i, ax in enumerate(np.argsort(-np.asarray(ext), kind="stable"))}

#### `_face_axis(face, ext)`
**Làm gì:** đổi tên mặt dạng `"C+"` thành `(chỉ số trục, dấu ±1)`.

Kiểm tra định dạng: 2 ký tự, ký tự đầu thuộc `ABC`, ký tự sau là `+` hoặc `-`; sai → `ValueError` với thông báo rõ
(bắt lỗi gõ nhầm trong manifest).

In [ ]:
def _face_axis(face: str, ext) -> tuple[int, int]:
    if len(face) != 2 or face[0] not in "ABC" or face[1] not in "+-":
        raise ValueError(f"tên mặt không hợp lệ: {face!r} (dạng 'C+', 'C-', 'A+', ...)")
    ax = next(a for a, n in _axis_names(ext).items() if n == face[0])
    return ax, (1 if face[1] == "+" else -1)

#### `_face_height(mesh, ax, sgn, px)`
**Làm gì:** ảnh độ cao nhìn thẳng vào **một mặt** của khối (trong khung OBB), dùng để chấm điểm mặt đó.

**Đầu vào:** mesh khung OBB, `ax` (trục), `sgn` (phía + hay −), `px` (bước lưới mm).
**Đầu ra:** mảng 2D độ cao (mm) theo hướng nhìn; ô không có dữ liệu = `−inf`.

**Cách làm:**
1. Chỉ giữ tam giác **quay mặt về phía người nhìn**: `n · d > 0,05` với `d` là hướng ra ngoài của mặt đang xét. Scan
   chỉ quét một mặt thì nhìn từ phía sau sẽ gần như trống — chính dấu hiệu `FACE_MISSING` dùng.
2. Rải (splat) các đỉnh và tâm tam giác đó lên lưới, mỗi ô giữ điểm cao nhất (`np.maximum.at`).

**Lưu ý:** splat điểm có thể để lọt lớp bề mặt bên dưới ở vài ô (xem `project_depth`). Chấp nhận được **ở đây** vì kết
quả chỉ dùng để lấy **trung vị** làm điểm số; không dùng hàm này để tạo ảnh.

In [ ]:
def _face_height(mesh: trimesh.Trimesh, ax: int, sgn: int, px: float) -> np.ndarray:
    """Z-buffer trực giao nhìn từ phía (ax, sgn) trong khung OBB: độ cao (mm) theo hướng nhìn, -inf = không có dữ liệu.
    Chỉ lấy tam giác quay mặt về phía người nhìn -> scan chỉ quét một mặt nhìn từ phía sau sẽ trống."""
    d = np.zeros(3); d[ax] = sgn
    sel = mesh.face_normals @ d > 0.05
    a1, a2 = [i for i in range(3) if i != ax]
    lo, ext = mesh.bounds[0], mesh.bounding_box.extents
    h = np.full((int(ext[a2] / px) + 1, int(ext[a1] / px) + 1), -np.inf, np.float32)
    if sel.any():
        P = np.vstack([mesh.vertices[np.unique(mesh.faces[sel])], mesh.triangles_center[sel]])
        ix = ((P[:, a1] - lo[a1]) / px).astype(int); iy = ((P[:, a2] - lo[a2]) / px).astype(int)
        np.maximum.at(h, (iy, ix), (sgn * P[:, ax]).astype(np.float32))
    return h

#### `_face_scores(h, px, long_mm)`
**Làm gì:** chấm điểm một mặt: mức độ chi tiết (có hoa văn / chữ khắc không) và độ phủ dữ liệu.

**Đầu vào:** ảnh độ cao `h` (từ `_face_height`), `px`, `long_mm` (cạnh dài nhất). **Đầu ra:** `{detail_mm, coverage}`.

**Độ phủ (`coverage`):** tỉ lệ ô có dữ liệu sau **phép đóng** hình thái (giãn 3 lần rồi co 3 lần, nhân 3 × 3). Phép
đóng lấp lỗ nhỏ giữa các điểm splat nhưng **không** làm mảnh vụn rời rạc phình to — dùng phép giãn đơn thuần thì
một mặt chỉ có vài mảnh vụn cũng được tính là phủ kín.

**Độ chi tiết (`detail_mm`):**
1. Lấp độ cao cho các ô vừa được phép đóng lấp (lan giá trị lân cận 3 lần).
2. **Lớp mặt ngoài:** giữ ô có `h > p90(h) − 0,075·L` (bỏ hốc sâu, vát cạnh), rồi co vào `2,5 %·L` (bỏ mép).
3. Phần dư cao tần: `res = h − G_σ * h` với `σ = 1 %·L`.
4. `detail = trung vị |res|` trên lớp mặt ngoài.

**Vì sao trung vị, không phải độ lệch chuẩn:** đã thử trên scan thật — độ lệch chuẩn chọn nhầm **mặt lưng** vì mép hai
hốc tay cầm rất sâu kéo nó lên; độ nhám bề mặt chọn nhầm một mặt cạnh. Trung vị chỉ cao khi chi tiết **phủ kín** mặt,
đúng đặc điểm của mặt khắc: 0,220 mm (mặt khắc) so với 0,025 mm (mặt lưng) trên scan mẫu.

In [ ]:
def _face_scores(h: np.ndarray, px: float, long_mm: float) -> dict:
    """detail_mm: trung vị |độ cao - bề mặt trơn (Gaussian σ = 1% cạnh dài)| trên lớp mặt ngoài -> mặt khắc có hoa
    văn/chữ phủ kín cho giá trị cao; mặt lưng phẳng (kể cả có vài hốc tay cầm lớn) cho giá trị thấp. Dùng TRUNG VỊ
    chứ không dùng độ lệch chuẩn: mép vài hốc sâu làm độ lệch chuẩn của mặt lưng vượt cả mặt khắc.
    coverage: phần diện tích mặt có bề mặt scan (sau khi lấp lỗ nhỏ giữa các điểm splat; mảnh vụn rời rạc
    không lấp kín được nên không được tính là có dữ liệu)."""
    raw = np.isfinite(h)
    k3 = np.ones((3, 3), np.uint8)
    # phép đóng: lấp lỗ nhỏ giữa các điểm splat mà không làm mảnh vụn phình to
    valid = cv2.erode(cv2.dilate(raw.astype(np.uint8), k3, iterations=3), k3, iterations=3).astype(bool)
    coverage = float(valid.mean())
    if raw.sum() < 100:
        return {"detail_mm": 0.0, "coverage": coverage}
    hf = np.where(raw, h, -1e9).astype(np.float32)
    for _ in range(3):                                   # gán độ cao cho các lỗ vừa lấp
        d = cv2.dilate(hf, k3)
        hf = np.where((hf < -1e8) & (d > -1e8), d, hf)
    valid &= hf > -1e8
    ref = np.percentile(hf[valid], 90)
    top = valid & (hf > ref - 0.075 * long_mm)          # lớp mặt ngoài: bỏ hốc sâu, vát cạnh
    er = max(1, int(0.025 * long_mm / px))              # co vào 2.5% cạnh dài, tránh mép
    top = cv2.erode(top.astype(np.uint8), np.ones((er, er), np.uint8)).astype(bool)
    if top.sum() < 100:
        return {"detail_mm": 0.0, "coverage": coverage}
    base = np.where(top, hf, np.median(hf[top])).astype(np.float32)
    res = base - cv2.GaussianBlur(base, (0, 0), 0.01 * long_mm / px)
    return {"detail_mm": float(np.median(np.abs(res[top]))), "coverage": coverage}

#### `detect_main_face(mesh)`
**Làm gì:** tự xác định mặt khắc của khối.

**Đầu vào:** mesh khung OBB (kết quả `prepare_scan`).
**Đầu ra:** dict gồm `face` (vd. `"C-"`), `confident`, `reason` (giải thích bằng lời), `ratio`, `flat`, `px_mm`,
`scores` (điểm từng mặt).

**Cách làm:**
1. `flat = C/B ≤ 0,6` → chỉ xét `C+`, `C-`; ngược lại xét cả 6 mặt.
2. Bước lưới `px = max(L/400, 0,7·√(2·S/N))` (S = diện tích bề mặt, N = số tam giác): đủ mịn nhưng không mịn hơn mật độ
   đỉnh của mesh, để tránh lỗ khi splat.
3. Chấm điểm từng mặt ứng viên (`_face_height` + `_face_scores`).
4. Chỉ **xếp hạng** các mặt có độ phủ ≥ 50 %; mặt được chọn = mặt có `detail` lớn nhất trong số đó.
5. Độ tin cậy:
   - mặt đối diện có độ phủ < 15 % → **tin cậy** (scan chỉ quét một mặt);
   - không còn mặt nào khác để so → **cần soát**;
   - `ratio = detail(chọn) / detail(thứ nhì) ≥ 2` → **tin cậy**, ngược lại **cần soát**.

`confident = False` không có nghĩa là sai — hàm vẫn trả mặt đoán tốt nhất, chỉ báo cần người xem ảnh soát.

In [ ]:
def detect_main_face(mesh: trimesh.Trimesh) -> dict:
    """Mesh ở khung OBB (prepare_scan) -> mặt khắc. Trả {face, auto, confident, reason, ratio, flat, px_mm, scores}.
    confident=False nghĩa là vẫn trả mặt đoán tốt nhất nhưng cần người soát (ảnh face_audit_image)."""
    ext = mesh.bounding_box.extents
    names = _axis_names(ext)
    long_mm = float(ext.max())
    e = np.sort(ext)
    flat = bool(e[0] / e[1] <= FACE_THIN_RATIO)
    axes = [a for a, n in names.items() if n == "C"] if flat else [0, 1, 2]
    # độ phân giải: 1/400 cạnh dài, nhưng không mịn hơn mật độ đỉnh của mesh (tránh lỗ khi splat)
    px = max(long_mm / 400, 0.7 * math.sqrt(2 * mesh.area / max(len(mesh.faces), 1)))
    scores = {}
    for ax in sorted(axes, key=lambda a: names[a]):
        for sgn in (1, -1):
            scores[names[ax] + ("+" if sgn > 0 else "-")] = _face_scores(_face_height(mesh, ax, sgn, px), px, long_mm)
    # chỉ xếp hạng các mặt có đủ dữ liệu: vài mảnh vụn (vd. mép cạnh thấy từ phía sau của scan hở) cho điểm chi
    # tiết cao giả tạo trên diện tích nhỏ
    def detail(k):
        return scores[k]["detail_mm"]
    elig = [k for k in scores if scores[k]["coverage"] >= FACE_MIN_COVERAGE]
    ratio = None
    if not elig:
        best = max(scores, key=lambda k: scores[k]["coverage"])
        confident, reason = False, f"không mặt nào có dữ liệu trên {FACE_MIN_COVERAGE:.0%} diện tích"
    else:
        best = max(elig, key=detail)
        opp = best[0] + ("-" if best[1] == "+" else "+")
        rest = [k for k in elig if k != best]
        if scores[opp]["coverage"] < FACE_MISSING:
            confident, reason = True, "scan chỉ quét một mặt (mặt đối diện không có dữ liệu)"
        elif not rest:
            confident, reason = False, "các mặt còn lại thiếu dữ liệu, không so sánh được"
        else:
            second = max(rest, key=detail)
            ratio = round(min(detail(best) / max(detail(second), 1e-9), 999.0), 2)
            confident = ratio >= FACE_MIN_RATIO
            reason = f"chi tiết gấp {ratio:.1f}x mặt thứ nhì ({second})" + ("" if confident else f", dưới ngưỡng {FACE_MIN_RATIO}")
    return {"face": best, "auto": True, "confident": confident, "reason": reason, "ratio": ratio,
            "flat": flat, "px_mm": round(px, 3),
            "scores": {k: {kk: round(vv, 4) for kk, vv in v.items()} for k, v in scores.items()}}

#### `orient_to_face(mesh, face, rot90)`
**Làm gì:** xoay mesh (khung OBB) sao cho mặt `face` hướng lên +Z — tư thế chuẩn để render và chiếu.

**Đầu vào:** mesh khung OBB, `face` (vd. `"C-"`), `rot90` (xoay thêm 0–3 × 90° quanh Z).
**Đầu ra:** bản sao mesh: mặt khắc hướng +Z, cạnh dài còn lại dọc trục X, đáy chạm `z = 0`, tâm XY ở gốc.

**Cách làm:** `z` = hướng ngoài của mặt được chọn; `x` = trục dài nhất trong hai trục còn lại; ma trận quay có các hàng
`[x, z × x, z]` — cách dựng này luôn cho det = +1 (không lật gương). Sau đó nhân phép xoay `rot90` quanh Z, rồi dời
để đáy ở `z = 0`.

**Metadata ghi lại:** `block_size_mm` (kích thước khối sau khi xoay), `to_render_frame` (ma trận từ toạ độ file gốc
sang khung render).

**Giới hạn:** hướng 0° hay 180° trong mặt phẳng (khối lộn đầu) **không suy ra được từ hình học** → chỉnh bằng `rot90`.

In [ ]:
def orient_to_face(mesh: trimesh.Trimesh, face: str, rot90: int = 0) -> trimesh.Trimesh:
    """Bản sao mesh (khung OBB) xoay cho `face` hướng +Z, cạnh dài còn lại nằm dọc trục X, rồi xoay thêm
    rot90 x 90° quanh Z; đáy chạm z=0, tâm XY ở gốc. Chỉ phép quay thật (det = +1).
    Hướng 0° / 180° trong mặt phẳng KHÔNG suy ra được từ hình học -> chỉnh bằng rot90 nếu cần."""
    ext = mesh.bounding_box.extents
    ax, sgn = _face_axis(face, ext)
    z = np.zeros(3); z[ax] = sgn
    xa = max((a for a in range(3) if a != ax), key=lambda a: ext[a])
    x = np.zeros(3); x[xa] = 1.0
    R = np.eye(4); R[:3, :3] = np.stack([x, np.cross(z, x), z])     # hàng = trục mới; y = z × x -> det = +1
    k = int(rot90) % 4
    c, s = (1, 0, -1, 0)[k], (0, 1, 0, -1)[k]
    Rz = np.eye(4); Rz[:2, :2] = [[c, -s], [s, c]]
    out = mesh.copy()
    out.apply_transform(Rz @ R)
    t = np.eye(4); t[:3, 3] = [-out.bounding_box.centroid[0], -out.bounding_box.centroid[1], -out.bounds[0][2]]
    out.apply_transform(t)
    out.metadata = dict(mesh.metadata)
    out.metadata["block_size_mm"] = [float(v) for v in out.bounding_box.extents]
    if "to_obb_frame" in mesh.metadata:
        out.metadata["to_render_frame"] = (t @ Rz @ R @ np.array(mesh.metadata["to_obb_frame"])).tolist()
    return out

#### `load_external_mesh(path, target_extent_mm, texture_path, face, rot90)`
**Làm gì:** gói trọn 3 bước `prepare_scan` → `detect_main_face` (hoặc mặt chỉ định tay) → `orient_to_face` trong một
lời gọi; ghi kết quả vào `metadata["face_info"]`.

Các công cụ chạy ở máy local (`run_smoke.py`, `check_face_detection.py`) dùng hàm này. Notebook **không** gọi nó ở
luồng chính mà tự làm 3 bước để **nạp mỗi scan đúng một lần** rồi dùng lại cho soát, phần A và phần B.

In [ ]:
def load_external_mesh(path: str | Path, target_extent_mm: float = 200.0, texture_path: str | Path | None = "auto",
                       face: str = "auto", rot90: int = 0) -> trimesh.Trimesh:
    """Nạp GLB/OBJ/STL/PLY -> mesh sẵn sàng render: mặt khắc hướng +Z, đáy z=0, cạnh dài nhất = target.
    face='auto' tự xác định (xem detect_main_face), hoặc chỉ định 'C+' / 'C-' / ... (tên trong ảnh soát).
    metadata['face_info'] ghi mặt đã chọn, lý do, độ tin cậy, điểm từng mặt."""
    m = prepare_scan(path, target_extent_mm, texture_path)
    info = detect_main_face(m) if face in (None, "", "auto") else \
        {"face": face, "auto": False, "confident": True, "reason": "chỉ định (tham số / manifest)"}
    info["rot90"] = int(rot90)
    out = orient_to_face(m, info["face"], rot90)
    out.metadata["face_info"] = info
    return out

#### `read_manifest(path)`
**Làm gì:** đọc `manifest.csv` — file chỉ định tay mặt khắc / texture cho từng scan.

**Cột dùng:** `mesh, texture, face, rot90` (cột khác bỏ qua, nên dùng thẳng được `audit.csv` do notebook sinh ra).
**Đầu ra:** dict `{tên file mesh: {"texture", "face", "rot90"}}`.

**Khoá là TÊN file mesh** (không kèm thư mục) để manifest viết ở máy này vẫn khớp trên Kaggle. Ô trống → `"auto"`
(tự xác định). Đọc bằng `utf-8-sig` để chịu được BOM do Excel thêm vào.

In [ ]:
def read_manifest(path: str | Path) -> dict[str, dict]:
    """CSV cột mesh, texture, face, rot90 (cột khác bỏ qua). Khoá = TÊN file mesh (không kèm thư mục) để dùng
    được ở máy khác / Kaggle. face trống hoặc 'auto' -> tự xác định; texture trống -> tự tìm.
    File audit.csv do audit_scans.py sinh ra có đúng các cột này: sửa dòng confident=False rồi dùng lại."""
    import csv
    out = {}
    with open(path, newline="", encoding="utf-8-sig") as f:
        for r in csv.DictReader(f):
            if not (r.get("mesh") or "").strip():
                continue
            out[Path(r["mesh"].strip()).name] = {
                "texture": (r.get("texture") or "").strip() or "auto",
                "face": (r.get("face") or "").strip() or "auto",
                "rot90": int(float(r.get("rot90") or 0)),
            }
    return out

### 3. Lấy mẫu camera / ánh sáng
Mô tả một lần chụp bằng ba dataclass (camera, đèn, cả lần chụp) và hàm lấy mẫu ngẫu nhiên các tham số đó theo 4 kiểu chụp thực tế.

#### Lớp `CameraSpec`
**Mô tả một camera.**

| Trường | Ý nghĩa |
|---|---|
| `elev_deg` | Góc cao của camera: 90° = nhìn thẳng từ trên xuống |
| `azim_deg` | Vị trí camera quanh khối (hướng phối cảnh) |
| `roll_deg` | Xoay trong mặt phẳng ảnh: 0° = trục +Y của khối hướng lên trên ảnh |
| `dist_mm` | Khoảng cách camera tới điểm nhìn |
| `yfov_deg` | Góc nhìn dọc (FOV) |
| `look_at` | Điểm camera nhìn vào (mm) |
| `width`, `height` | Kích thước ảnh (px) |

In [ ]:
@dataclass
class CameraSpec:
    elev_deg: float      # 90 = nhìn thẳng từ trên xuống
    azim_deg: float      # vị trí camera quanh khối (hướng phối cảnh)
    roll_deg: float      # xoay trong mặt phẳng ảnh: 0 = trục +Y của khối hướng lên, 90/180 = khối bị xoay trong khung
    dist_mm: float
    yfov_deg: float
    look_at: list[float]
    width: int
    height: int

#### Lớp `LightSpec`
**Mô tả một nguồn sáng.** `kind`: `directional` (đèn xa, song song), `point` (đèn điểm, vd. flash), `spot`;
`elev_deg`, `azim_deg`: hướng tới đèn; `intensity`; `color` (RGB 0–1, từ nhiệt độ màu); `dist_mm` (khoảng cách, chỉ có
ý nghĩa với point/spot).

In [ ]:
@dataclass
class LightSpec:
    kind: str            # 'directional' | 'point' | 'spot'
    elev_deg: float
    azim_deg: float
    intensity: float
    color: list[float]
    dist_mm: float = 600.0

#### Lớp `ShotSpec`
**Mô tả trọn một lần chụp:** camera + danh sách đèn + ánh sáng môi trường (`ambient`, RGB) + dict tham số hậu kỳ
(`post`). Được ghi nguyên vào `meta.json` của mỗi ảnh (qua `asdict`) để tái tạo và truy vết.

In [ ]:
@dataclass
class ShotSpec:
    camera: CameraSpec
    lights: list[LightSpec]
    ambient: list[float]
    post: dict = field(default_factory=dict)

#### `_kelvin_rgb(k)`
**Làm gì:** đổi nhiệt độ màu (Kelvin) sang màu RGB của nguồn sáng.

**Cách làm:** xấp xỉ đường cong của Tanner Helland, với `t = K/100`:
- đỏ = 255 khi `t ≤ 66`, ngược lại `329,7·(t−60)^−0,1332`;
- lục = `99,47·ln t − 161,1` khi `t ≤ 66`, ngược lại `288,1·(t−60)^−0,0755`;
- lam = 255 khi `t ≥ 66`, 0 khi `t ≤ 19`, ngược lại `138,5·ln(t−10) − 305`.

Kẹp về [0, 1]. Ví dụ 3000 K ra vàng cam (đèn sợi đốt), 6500 K gần trắng (ánh sáng ban ngày).

In [ ]:
def _kelvin_rgb(k: float) -> list[float]:
    """Xấp xỉ màu nguồn sáng theo nhiệt độ màu (Kelvin) -> RGB [0..1]."""
    t = k / 100.0
    r = 255 if t <= 66 else 329.7 * ((t - 60) ** -0.1332)
    g = 99.47 * math.log(t) - 161.1 if t <= 66 else 288.1 * ((t - 60) ** -0.0755)
    b = 255 if t >= 66 else (0 if t <= 19 else 138.5 * math.log(t - 10) - 305.0)
    return [float(np.clip(v / 255, 0, 1)) for v in (r, g, b)]

#### `sample_shot(rng, block_wh_mm, width, height, preset)`
**Làm gì:** lấy mẫu ngẫu nhiên **một lần chụp** (camera + đèn + hậu kỳ) theo một kiểu chụp thực tế.

**Đầu vào:** `rng` (bộ sinh số ngẫu nhiên có seed → tái tạo được), `block_wh_mm` (kích thước khối), kích thước ảnh,
`preset`. **Đầu ra:** `ShotSpec`.

**Chọn preset:** `mixed` bốc ngẫu nhiên topdown 25 %, handheld 25 %, raking 20 %, closeup 30 %.

| Preset | `elev` | FOV | Khối chiếm khung | Đèn |
|---|---|---|---|---|
| topdown (chụp lưu trữ) | 78–90° | 30–45° | 0,75–0,95 | directional cao 50–80°, ambient 0,25–0,5 |
| handheld (chụp tay) | 45–78° | 40–65° | 0,7–1,0 | 50 % flash gần camera, 50 % đèn phòng; 40 % thêm nguồn phụ 6500 K (cửa sổ) |
| raking (đèn xiên) | 55–88° | 32–50° | 0,75–1,0 | directional **thấp 10–30°** → bóng dài, nét nổi rõ |
| closeup (cận cảnh) | 60–90° | 30–45° | **2–4×** (chỉ thấy một phần khối) | directional 25–80°, 50 % có nguồn phụ |

**Chung cho mọi preset:**
- azimuth đều trong 0–360°;
- roll: 80 % gần thẳng `N(0°, 4°)`, 20 % xoay 90/180/270° (ảnh chụp vội);
- khoảng cách `dist = (đường chéo khối / fill) / (2·tan(FOV/2))` để khối chiếm đúng tỉ lệ `fill` của chiều cao khung;
- điểm nhìn lệch nhẹ khỏi tâm (closeup lệch tới ±35 % kích thước khối);
- nhiệt độ màu 3000 / 4000 / 5000 / 5600 / 6500 K;
- hậu kỳ: EV −0,4…0,5; gamma 0,9–1,15; nhiễu 0–0,03; mờ `max(0, N(0,3; 0,4))` px; vignette 0–0,35; JPEG 55–95;
  lệch cân bằng trắng R/B ±5 %; màu mặt bàn chọn trong 5 màu (giấy trắng, gỗ, vải xám, nền tối, be).

In [ ]:
def sample_shot(rng: random.Random, block_wh_mm: tuple[float, float], width=1024, height=768,
                preset: str = "mixed") -> ShotSpec:
    """
    preset:
      'topdown'  : chụp gần thẳng, đèn khuếch tán (như chụp lưu trữ)
      'handheld' : góc nghiêng, đèn flash gần camera hoặc đèn phòng
      'raking'   : đèn xiên thấp -> bóng dài, relief nổi rõ
      'mixed'    : chọn ngẫu nhiên các preset trên
    """
    if preset == "mixed":
        preset = rng.choices(["topdown", "handheld", "raking", "closeup"], weights=[0.25, 0.25, 0.2, 0.3])[0]
    bw, bh = block_wh_mm
    diag = math.hypot(bw, bh)

    # elev: 90 = nhìn thẳng từ trên; azim ưu tiên 4 hướng trục (khối gần thẳng trong khung) + jitter
    azim = rng.uniform(0, 360)
    # hướng khối trong khung: 80% gần thẳng (jitter nhỏ), 20% xoay 90/180/270 như ảnh chụp vội
    u = rng.random()
    if u < 0.8:
        roll = rng.gauss(0, 4)
    else:
        roll = rng.choice([90, 180, 270]) + rng.gauss(0, 4)
    if preset == "topdown":
        elev = rng.uniform(78, 90); yfov = rng.uniform(30, 45); fill = rng.uniform(0.75, 0.95)
    elif preset == "handheld":
        elev = rng.uniform(45, 78); yfov = rng.uniform(40, 65); fill = rng.uniform(0.7, 1.0)
    elif preset == "raking":
        elev = rng.uniform(55, 88); yfov = rng.uniform(32, 50); fill = rng.uniform(0.75, 1.0)
    else:  # closeup: chỉ thấy một phần khối
        elev = rng.uniform(60, 90); yfov = rng.uniform(30, 45); fill = rng.uniform(2.0, 4.0)
    # khoảng cách sao cho đường chéo khối chiếm ~fill lần chiều cao khung
    dist = (diag / fill) / (2 * math.tan(math.radians(yfov / 2)))
    if preset == "closeup":
        look = [rng.uniform(-bw * 0.35, bw * 0.35), rng.uniform(-bh * 0.35, bh * 0.35), 0.0]
    else:
        look = [rng.gauss(0, bw * 0.04), rng.gauss(0, bh * 0.04), 0.0]
    cam = CameraSpec(elev, azim, roll, dist, yfov, look, width, height)

    lights: list[LightSpec] = []
    kelvin = rng.choice([3000, 4000, 5000, 5600, 6500])
    col = _kelvin_rgb(kelvin)
    if preset == "topdown":
        lights.append(LightSpec("directional", rng.uniform(50, 80), rng.uniform(0, 360), rng.uniform(2.0, 4.0), col))
        ambient = [rng.uniform(0.25, 0.5)] * 3
    elif preset == "handheld":
        if rng.random() < 0.5:  # flash gần camera
            lights.append(LightSpec("point", elev + rng.uniform(-8, 8), azim + rng.uniform(-15, 15),
                                    rng.uniform(4e5, 9e5), col, dist_mm=dist * 0.95))
        else:                   # đèn phòng
            lights.append(LightSpec("directional", rng.uniform(35, 75), rng.uniform(0, 360), rng.uniform(1.5, 3.5), col))
        if rng.random() < 0.4:  # thêm nguồn phụ màu khác (cửa sổ)
            lights.append(LightSpec("directional", rng.uniform(20, 60), rng.uniform(0, 360), rng.uniform(0.5, 1.5), _kelvin_rgb(6500)))
        ambient = [rng.uniform(0.1, 0.35)] * 3
    elif preset == "raking":
        lights.append(LightSpec("directional", rng.uniform(10, 30), rng.uniform(0, 360), rng.uniform(3.0, 6.0), col))
        ambient = [rng.uniform(0.12, 0.3)] * 3
    else:  # closeup: đèn bất kỳ, thường có 2 nguồn
        lights.append(LightSpec("directional", rng.uniform(25, 80), rng.uniform(0, 360), rng.uniform(2.0, 4.5), col))
        if rng.random() < 0.5:
            lights.append(LightSpec("directional", rng.uniform(20, 60), rng.uniform(0, 360), rng.uniform(0.5, 1.5), _kelvin_rgb(rng.choice([3000, 6500]))))
        ambient = [rng.uniform(0.15, 0.4)] * 3

    post = {
        "exposure": rng.uniform(-0.4, 0.5),
        "gamma": rng.uniform(0.9, 1.15),
        "noise_sigma": rng.uniform(0.0, 0.03),
        "blur_sigma": max(0.0, rng.gauss(0.3, 0.4)),
        "vignette": rng.uniform(0.0, 0.35),
        "jpeg_q": int(rng.uniform(55, 95)),
        "wb_shift": [rng.uniform(0.95, 1.05), 1.0, rng.uniform(0.95, 1.05)],
        # màu mặt bàn: giấy trắng (như ảnh lưu trữ), gỗ, vải xám, nền tối
        "table_rgb": rng.choice([[0.92, 0.91, 0.88], [0.55, 0.42, 0.30], [0.45, 0.45, 0.47], [0.15, 0.15, 0.15], [0.75, 0.72, 0.66]]),
    }
    return ShotSpec(cam, lights, ambient, post | {"preset": preset, "kelvin": kelvin})

#### `frontal_shot(block_size_mm, width, height, fill, yfov_deg)`
**Làm gì:** một lần chụp **cố định** nhìn thẳng mặt +Z: camera `elev = 90°`, đèn directional 40° / 135°, ambient 0,3,
không hậu kỳ. Dùng cho **ảnh soát mặt khắc**, để mọi mặt được chụp y như nhau và so sánh được bằng mắt.

Khoảng cách tính để khối chiếm `fill = 0,9` khung, cộng thêm độ dày khối để camera không cắt vào khối.

In [ ]:
def frontal_shot(block_size_mm, width: int, height: int, fill: float = 0.9, yfov_deg: float = 35.0) -> ShotSpec:
    """Nhìn thẳng mặt +Z (ảnh: +X sang phải, +Y lên), đèn xiên 40° cố định, không hậu kỳ -> để soát mặt chính."""
    bw, bh, bz = block_size_mm
    dist = (math.hypot(bw, bh) / fill) / (2 * math.tan(math.radians(yfov_deg / 2))) + bz
    return ShotSpec(CameraSpec(90.0, 0.0, 0.0, dist, yfov_deg, [0.0, 0.0, 0.0], width, height),
                    [LightSpec("directional", 40.0, 135.0, 3.5, [1.0, 1.0, 1.0])], [0.3, 0.3, 0.3],
                    {"table_rgb": [0.9, 0.9, 0.88]})

### 4. Toán học camera
Chuyển góc (elev, azim, roll) thành vector hướng và ma trận pose camera kiểu OpenGL.

#### `_spherical_dir(elev_deg, azim_deg)`
**Làm gì:** vector đơn vị ứng với góc cao `elev` và phương vị `azim`:
`(cos e·cos a, cos e·sin a, sin e)`. `elev = 90°` → `(0, 0, 1)` (thẳng lên).

Dùng cho vị trí camera và hướng đèn. Phần B có một bản giống hệt để hai phần dùng chung quy ước góc.

In [ ]:
def _spherical_dir(elev_deg: float, azim_deg: float) -> np.ndarray:
    e, a = math.radians(elev_deg), math.radians(azim_deg)
    return np.array([math.cos(e) * math.cos(a), math.cos(e) * math.sin(a), math.sin(e)])

#### `look_at_pose(eye, target, roll_deg)`
**Làm gì:** ma trận pose 4 × 4 của camera kiểu OpenGL (camera nhìn theo trục −Z của chính nó), đặt ở `eye`, nhìn vào
`target`, xoay `roll_deg` trong mặt phẳng ảnh.

**Cách làm:**
- `f = normalize(target − eye)` (hướng nhìn);
- hướng "lên" gợi ý = trục +Y của khối xoay `roll` quanh Z: `(−sin r, cos r, 0)`;
- `s = normalize(f × up)` (phải), `u = s × f` (lên thật);
- các cột của ma trận: `s`, `u`, `−f`, `eye`.

**Vì sao gắn "lên" với trục +Y của khối:** dùng trục Z thế giới làm "lên" như thường lệ thì khi camera nhìn gần thẳng
xuống (`elev ≈ 80–90°`) hướng ảnh bị nhảy đột ngột. Gắn với trục dọc của khối thì khối luôn đứng khi `roll = 0`, ở mọi
góc. Nhìn dọc đúng trục +Y (suy biến) → dùng +Z làm dự phòng.

In [ ]:
def look_at_pose(eye: np.ndarray, target: np.ndarray, roll_deg: float = 0.0) -> np.ndarray:
    """
    Ma trận pose 4x4 (OpenGL: camera nhìn theo -Z của chính nó).
    Hướng "lên" của ảnh = trục +Y thế giới (trục dọc của khối) xoay roll_deg quanh Z, chiếu vuông góc
    với hướng nhìn -> khối luôn đứng khi roll=0, bất kể elev/azim (không nhảy hướng ở elev ~80°).
    """
    f = target - eye; f /= np.linalg.norm(f)
    r = math.radians(roll_deg)
    up_hint = np.array([-math.sin(r), math.cos(r), 0.0])
    s = np.cross(f, up_hint)
    if np.linalg.norm(s) < 1e-6:  # nhìn dọc theo trục +Y (elev ~0): fallback +Z
        s = np.cross(f, np.array([0.0, 0.0, 1.0]))
    s /= np.linalg.norm(s)
    u = np.cross(s, f)
    M = np.eye(4)
    M[:3, 0] = s; M[:3, 1] = u; M[:3, 2] = -f; M[:3, 3] = eye
    return M

#### `camera_pose_from_spec(c)`
**Làm gì:** từ `CameraSpec` tính vị trí camera `eye = look_at + _spherical_dir(elev, azim) · dist` rồi dựng pose bằng
`look_at_pose`.

In [ ]:
def camera_pose_from_spec(c: CameraSpec) -> np.ndarray:
    eye = np.array(c.look_at) + _spherical_dir(c.elev_deg, c.azim_deg) * c.dist_mm
    return look_at_pose(eye, np.array(c.look_at), c.roll_deg)

### 5. Renderer pyrender
Bọc pyrender: dựng cảnh (khối + mặt bàn + camera + đèn) và render ra ảnh màu + độ sâu, có bóng đổ.

#### Lớp `PyrenderBackend`
**Làm gì:** bọc pyrender để render một khối nhiều lần với các lần chụp khác nhau.

**`__init__(mesh, width, height)`:**
- chuyển mesh sang pyrender (`smooth=False`: pháp tuyến theo từng tam giác, giữ nét sắc của mép khắc);
- vật liệu gỗ thấm mực: roughness 0,6 (hơi bóng), metallic 0;
- tạo `OffscreenRenderer` (EGL trên Kaggle);
- tạo **mặt bàn** rộng gấp 4 lần khối, dày 1 mm, ngay dưới đáy khối để nhận bóng đổ.

**`render(shot)`:** dựng cảnh mới mỗi lần (ambient; nền và mặt bàn cùng màu `table_rgb`), camera phối cảnh
(`znear` 5 mm, `zfar` 5000 mm), thêm từng đèn theo hướng trong `LightSpec`, render có **bóng đổ** cho đèn directional
và spot. Trả `(ảnh màu, ảnh độ sâu, pose camera)`. Mesh chỉ nạp lên GPU một lần và được dùng lại giữa các cảnh.

**`gl_info()`:** tên GPU / driver OpenGL thật sự đang render. `llvmpipe`, `softpipe`, `SwiftShader` nghĩa là EGL
**không** tới được driver GPU và đang render bằng CPU — chậm hàng chục lần dù máy có GPU. Cell cấu hình in thông tin này.

**`close()`:** giải phóng renderer (bộ nhớ GPU).

In [ ]:
class PyrenderBackend:
    def __init__(self, mesh: trimesh.Trimesh, width: int, height: int):
        import pyrender
        self.pr = pyrender
        self.width, self.height = width, height
        self.mesh_node_mesh = pyrender.Mesh.from_trimesh(mesh, smooth=False)
        for prim in self.mesh_node_mesh.primitives:   # gỗ thấm mực: hơi bóng
            prim.material.roughnessFactor = 0.6
            prim.material.metallicFactor = 0.0
        self.r = pyrender.OffscreenRenderer(viewport_width=width, viewport_height=height)
        # mặt bàn phía dưới khối để có bóng đổ ra ngoài
        bw, bh = mesh.metadata["block_size_mm"][:2]
        table = trimesh.creation.box(extents=[bw * 4, bh * 4, 1.0])
        table.apply_translation([0, 0, -0.5])
        table.visual = trimesh.visual.ColorVisuals(table, face_colors=[120, 110, 100, 255])
        self.table = pyrender.Mesh.from_trimesh(table, smooth=False)

    def render(self, shot: ShotSpec) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        pr = self.pr
        table_col = shot.post.get("table_rgb", [0.5, 0.48, 0.45])
        scene = pr.Scene(ambient_light=np.array(shot.ambient + [1.0]), bg_color=table_col + [1.0])
        scene.add(self.mesh_node_mesh)
        self.table.primitives[0].material.baseColorFactor = np.array(table_col + [1.0])
        scene.add(self.table)
        c = shot.camera
        cam = pr.PerspectiveCamera(yfov=math.radians(c.yfov_deg), aspectRatio=c.width / c.height, znear=5.0, zfar=5000.0)
        cam_pose = camera_pose_from_spec(c)
        scene.add(cam, pose=cam_pose)
        for L in shot.lights:
            d = _spherical_dir(L.elev_deg, L.azim_deg)
            pose = look_at_pose(d * L.dist_mm, np.zeros(3))
            if L.kind == "directional":
                scene.add(pr.DirectionalLight(color=L.color, intensity=L.intensity), pose=pose)
            elif L.kind == "point":
                scene.add(pr.PointLight(color=L.color, intensity=L.intensity), pose=pose)
            else:
                scene.add(pr.SpotLight(color=L.color, intensity=L.intensity,
                                       innerConeAngle=0.3, outerConeAngle=0.6), pose=pose)
        flags = pr.RenderFlags.SHADOWS_DIRECTIONAL | pr.RenderFlags.SHADOWS_SPOT
        color, depth = self.r.render(scene, flags=flags)
        return color, depth, cam_pose

    def gl_info(self) -> dict:
        """GPU / driver OpenGL THẬT SỰ đang render. 'llvmpipe' / 'softpipe' / 'SwiftShader' = render bằng CPU
        (EGL không tới được driver GPU) -> chậm hàng chục lần dù máy có GPU."""
        from OpenGL.GL import glGetString, GL_RENDERER, GL_VENDOR, GL_VERSION
        self.r._platform.make_current()
        return {k: (glGetString(v) or b"").decode(errors="replace")
                for k, v in (("renderer", GL_RENDERER), ("vendor", GL_VENDOR), ("version", GL_VERSION))}

    def close(self):
        self.r.delete()

### 6. Hậu kỳ giả camera
Biến ảnh render "sạch" thành ảnh giống máy ảnh thật: phơi sáng, cân bằng trắng, gamma, tối góc, mờ, nhiễu cảm biến, nén JPEG.

#### `apply_post(rgb, post, rng, min_mean)`
**Làm gì:** biến ảnh render sạch thành ảnh giống máy ảnh thật. Các bước theo đúng thứ tự:

| Bước | Công thức |
|---|---|
| Phơi sáng | `img · 2^EV`. **Auto-exposure tối thiểu:** nếu độ sáng trung bình sau phơi sáng < `min_mean` (0,14) thì tăng EV cho đủ, như máy ảnh tự nâng ISO — tránh ảnh gần đen hoàn toàn (hay gặp ở đèn xiên thấp) |
| Cân bằng trắng | nhân từng kênh với `wb_shift` |
| Gamma | `img^γ` |
| Vignette (tối góc) | `img · (1 − v·r²)`, `r` = khoảng cách chuẩn hoá tới tâm ảnh |
| Mờ | Gauss σ = `blur_sigma` (bỏ qua nếu < 0,05) |
| Nhiễu cảm biến | Poisson-Gauss: `σ = s·(0,6 + 1,2·√(1 − L))` — **vùng tối nhiễu mạnh hơn** như ảnh ISO cao; thêm nhiễu màu theo kênh ở vùng `L < 0,3` |
| Nén JPEG | mã hoá rồi giải mã lại với chất lượng `jpeg_q` → có đúng vết khối 8 × 8 của JPEG thật |

EV thực dùng được ghi lại vào `post["exposure_applied"]` khi auto-exposure can thiệp.

In [ ]:
def apply_post(rgb: np.ndarray, post: dict, rng: random.Random, min_mean: float = 0.14) -> np.ndarray:
    img = rgb.astype(np.float32) / 255.0
    exposure = post.get("exposure", 0.0)
    # auto-exposure tối thiểu: máy ảnh thật sẽ tự tăng ISO/phơi sáng, không cho ảnh gần đen hoàn toàn
    mean = float(img.mean()) * (2.0 ** exposure)
    if mean < min_mean:
        exposure += math.log2(min_mean / max(mean, 1e-4))
        post["exposure_applied"] = exposure
    img = img * (2.0 ** exposure)
    img = img * np.array(post.get("wb_shift", [1, 1, 1]), dtype=np.float32)[None, None, :]
    img = np.clip(img, 0, 1) ** post.get("gamma", 1.0)
    v = post.get("vignette", 0.0)
    if v > 0:
        h, w = img.shape[:2]
        yy, xx = np.mgrid[0:h, 0:w]
        r = np.sqrt(((xx - w / 2) / (w / 2)) ** 2 + ((yy - h / 2) / (h / 2)) ** 2)
        img *= (1 - v * np.clip(r, 0, 1.2) ** 2)[..., None]
    b = post.get("blur_sigma", 0.0)
    if b > 0.05:
        img = cv2.GaussianBlur(img, (0, 0), b)
    s = post.get("noise_sigma", 0.0)
    if s > 0:  # Poisson-Gaussian: vùng tối nhiễu mạnh hơn (ISO cao), thêm nhiễu màu theo kênh ở vùng tối
        nrng = np.random.default_rng(rng.randrange(1 << 30))
        lum = img.mean(-1, keepdims=True)
        sig = s * (0.6 + 1.2 * np.sqrt(np.clip(1 - lum, 0, 1)))
        img = img + nrng.normal(0, 1, img.shape).astype(np.float32) * sig
        img = img + nrng.normal(0, s * 0.4, (1, 1, 3)).astype(np.float32) * (lum < 0.3)
    out = (np.clip(img, 0, 1) * 255).astype(np.uint8)
    q = int(post.get("jpeg_q", 95))
    ok, buf = cv2.imencode(".jpg", cv2.cvtColor(out, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, q])
    return cv2.cvtColor(cv2.imdecode(buf, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)

### 6b. Che khuất bởi vật ngoài (tầng 2D, sau render)
Vẽ thêm vật che thường gặp khi chụp mộc bản lên ảnh đã render.

#### `add_occluders(img, rng, prob)`
**Làm gì:** vẽ thêm vật che lên ảnh đã render (tầng 2D).

**Đầu vào:** ảnh RGB, `rng`, `prob` (xác suất ảnh có vật che). **Đầu ra:** `(ảnh, mask che 0/1, danh sách loại)`.

Với xác suất `prob`, thêm 1–2 vật (trọng số ngón tay 3 : thước 2 : băng dính 2 : lóa 2 : bóng 2):

| Loại | Cách vẽ | Tính vào mask che? |
|---|---|---|
| Ngón tay | 1–2 ellipse màu da (3 tông) từ mép trái / phải / dưới, kèm bóng mềm | có |
| Thước | dải vàng hoặc trắng dọc / ngang gần mép | có |
| Băng dính | chữ nhật xanh hoặc trắng | có |
| Lóa flash | đốm trắng mềm, cộng sáng | **không** — bề mặt vẫn thấy một phần |
| Bóng người chụp | dải tối mềm hình thang | **không** |

Mask che cho phép tính `occluded_frac` (phần trăm diện tích bị che) ghi vào meta, dùng để lọc nhãn sau này.

In [ ]:
def add_occluders(img: np.ndarray, rng: random.Random, prob: float = 0.5) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """
    Thêm vật che thường gặp khi chụp mộc bản: ngón tay/bàn tay, thước, nhãn băng dính, lóa flash, bóng người chụp.
    Trả về (ảnh, mask che 0/1 HxW, danh sách loại đã thêm). Lóa/bóng không tính vào mask che (bề mặt vẫn thấy được một phần).
    """
    h, w = img.shape[:2]
    out = img.astype(np.float32) / 255
    occ = np.zeros((h, w), np.float32)
    kinds: list[str] = []
    if rng.random() >= prob:
        return img, occ, kinds
    n = rng.choice([1, 1, 2])
    for _ in range(n):
        k = rng.choices(["finger", "ruler", "tape", "glare", "shadow"], weights=[3, 2, 2, 2, 2])[0]
        kinds.append(k)
        layer = np.zeros((h, w), np.float32)
        if k == "finger":  # 1-2 ngón từ mép ảnh
            side = rng.choice(["l", "r", "b"])
            col = np.array(rng.choice([[0.85, 0.65, 0.5], [0.72, 0.5, 0.36], [0.55, 0.36, 0.25]]), np.float32)
            for j in range(rng.choice([1, 2])):
                fw = int(rng.uniform(0.05, 0.09) * w); fl = int(rng.uniform(0.2, 0.45) * h)
                if side == "b":
                    cx = int(rng.uniform(0.15, 0.85) * w) + j * int(fw * 1.3)
                    cv2.ellipse(layer, (cx, h), (fw // 2, fl), 0, 0, 360, 1.0, -1)
                else:
                    cy = int(rng.uniform(0.2, 0.8) * h) + j * int(fw * 1.3); cx = 0 if side == "l" else w
                    cv2.ellipse(layer, (cx, cy), (fl, fw // 2), 0, 0, 360, 1.0, -1)
            shade = cv2.GaussianBlur(layer, (0, 0), 6) * 0.35   # bóng mềm quanh ngón
        elif k == "ruler":  # thước vàng/trắng nằm dọc hoặc ngang gần mép
            t = int(rng.uniform(0.05, 0.09) * min(h, w))
            vertical = rng.random() < 0.5
            pos = int(rng.uniform(0.02, 0.15) * (w if vertical else h))
            if vertical:
                layer[:, pos:pos + t] = 1
            else:
                layer[pos:pos + t, :] = 1
            col = np.array(rng.choice([[0.95, 0.85, 0.2], [0.95, 0.95, 0.9]]), np.float32)
            shade = np.zeros_like(layer)
        elif k == "tape":  # nhãn băng dính xanh/trắng
            tw, th_ = int(rng.uniform(0.08, 0.18) * w), int(rng.uniform(0.03, 0.06) * h)
            x, y = int(rng.uniform(0, w - tw)), int(rng.uniform(0, h - th_))
            layer[y:y + th_, x:x + tw] = 1
            col = np.array(rng.choice([[0.3, 0.55, 0.9], [0.95, 0.95, 0.92]]), np.float32)
            shade = np.zeros_like(layer)
        elif k == "glare":  # lóa flash: đốm trắng bão hoà mềm (không tính che)
            cx, cy = int(rng.uniform(0.2, 0.8) * w), int(rng.uniform(0.2, 0.8) * h)
            r = int(rng.uniform(0.12, 0.3) * min(h, w))
            cv2.circle(layer, (cx, cy), r, 1.0, -1)
            g = cv2.GaussianBlur(layer, (0, 0), r * 0.5)[..., None] * rng.uniform(0.5, 0.9)
            out = np.clip(out + g, 0, 1)
            continue
        else:  # bóng người chụp: dải tối mềm (không tính che)
            x0 = int(rng.uniform(-0.2, 0.6) * w); x1 = x0 + int(rng.uniform(0.3, 0.7) * w)
            pts = np.array([[x0, 0], [x1, 0], [x1 + int(0.2 * w), h], [x0 + int(0.2 * w), h]], np.int32)
            cv2.fillPoly(layer, [pts], 1.0)
            d = cv2.GaussianBlur(layer, (0, 0), 25)[..., None] * rng.uniform(0.3, 0.6)
            out = out * (1 - d)
            continue
        a = layer[..., None]
        out = out * (1 - a) + col[None, None, :] * a
        out = out * (1 - shade[..., None])
        occ = np.maximum(occ, layer)
    return (np.clip(out, 0, 1) * 255).astype(np.uint8), occ, kinds

### 7. Vòng lặp sinh dữ liệu
Ghép các bước trên thành vòng sinh ảnh hàng loạt, cùng các hàm soát mặt khắc và ghép ảnh xem nhanh.

#### `render_dataset(backend, out_dir, n, block_wh_mm, width, height, preset, seed, name, occluder_prob, extra_meta)`
**Làm gì:** vòng sinh ảnh hàng loạt cho một khối.

**Đầu vào:** backend đã tạo, thư mục ra, số ảnh `n`, kích thước khối, kích thước ảnh, `preset`, `seed`, tên file,
`occluder_prob`, `extra_meta` (thêm vào mọi bản ghi, vd. mặt khắc đã chọn).

**Mỗi ảnh:** `sample_shot` → `render` → `apply_post` → `add_occluders` → (nếu có vật che: nén JPEG lại chất lượng 90 để
vật che cũng có vết nén như phần còn lại) → ghi `<tên>_XXXX.jpg` (chất lượng 95).

**Ghi `<tên>_meta.json`:** mỗi ảnh một bản ghi gồm `file`, `shot` (toàn bộ tham số), `cam_pose` (ma trận 4 × 4),
`occluders`, `occluded_frac`, cộng `extra_meta`. Cùng `seed` → cùng bộ ảnh.

In [ ]:
def render_dataset(
    backend,
    out_dir: str | Path,
    n: int,
    block_wh_mm: tuple[float, float],
    width: int = 1024,
    height: int = 768,
    preset: str = "mixed",
    seed: int = 0,
    name: str = "model",
    occluder_prob: float = 0.0,
    extra_meta: Optional[dict] = None,
) -> list[dict]:
    """
    Render n ảnh: mỗi ảnh lấy mẫu góc chụp/đèn (sample_shot) -> render -> hậu kỳ camera -> vật che 2D.
    Ghi <name>_XXXX.jpg + <name>_meta.json (pose camera, đèn, tham số hậu kỳ, vật che, % diện tích bị che).
    """
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    rng = random.Random(seed)
    records = []
    for i in range(n):
        shot = sample_shot(rng, block_wh_mm, width, height, preset)
        color, depth, cam_pose = backend.render(shot)
        img = apply_post(color, shot.post, rng)
        img, occ, kinds = add_occluders(img, rng, occluder_prob)
        if kinds:  # vật che thêm sau JPEG của apply_post -> nén lại nhẹ cho đồng nhất
            _, buf = cv2.imencode(".jpg", cv2.cvtColor(img, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 90])
            img = cv2.cvtColor(cv2.imdecode(buf, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        fn = f"{name}_{i:04d}.jpg"
        cv2.imwrite(str(out_dir / fn), cv2.cvtColor(img, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 95])
        rec = {"file": fn, "shot": asdict(shot), "cam_pose": cam_pose.tolist(), "occluders": kinds,
               "occluded_frac": round(float((occ >= 0.5).mean()), 4)}
        if extra_meta:
            rec.update(extra_meta)
        records.append(rec)
    with open(out_dir / f"{name}_meta.json", "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=1)
    return records

#### `face_audit_image(mesh_obb, info, width, height)`
**Làm gì:** ảnh soát mặt khắc: render nhìn thẳng (`frontal_shot`) **từng mặt ứng viên** rồi ghép thành lưới tối đa
3 cột, mỗi ô ghi `detail` và `coverage` của mặt đó.

**Khung màu ở mặt được chọn:** **xanh** = tự chọn và tin cậy; **vàng** = tự chọn nhưng cần người xem; **tím** = chỉ
định tay.

Cần pyrender (GPU). Mỗi mặt tạo một renderer riêng nên tốn vài giây mỗi mặt.

In [ ]:
def face_audit_image(mesh_obb: trimesh.Trimesh, info: dict, width: int = 480, height: int = 360) -> np.ndarray:
    """Ảnh soát (RGB): nhìn thẳng từng mặt ứng viên kèm điểm chi tiết / độ phủ. Khung XANH = mặt được chọn và tin
    cậy, VÀNG = được chọn nhưng cần người xem, TÍM = chỉ định tay. mesh_obb: kết quả prepare_scan."""
    scores = info.get("scores") or {info["face"]: None}
    tiles = []
    for name, sc in scores.items():
        m = orient_to_face(mesh_obb, name)
        be = PyrenderBackend(m, width, height)
        img, _, _ = be.render(frontal_shot(m.metadata["block_size_mm"], width, height))
        be.close()
        img = np.ascontiguousarray(img)
        if name == info["face"]:
            col = (170, 60, 220) if not info.get("auto", True) else (0, 190, 0) if info["confident"] else (255, 185, 0)
            cv2.rectangle(img, (0, 0), (width - 1, height - 1), col, 10)
        txt = name if sc is None else f"{name}  detail {sc['detail_mm']:.3f}mm  cover {sc['coverage'] * 100:.0f}%"
        cv2.putText(img, txt, (12, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (20, 20, 20), 4)
        cv2.putText(img, txt, (12, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (255, 255, 255), 1)
        tiles.append(img)
    cols = min(len(tiles), 3)
    while len(tiles) % cols:
        tiles.append(np.zeros_like(tiles[0]))
    return np.vstack([np.hstack(tiles[i:i + cols]) for i in range(0, len(tiles), cols)])

#### `audit_scan(path, target_extent_mm, override, width, height, image)`
**Làm gì:** soát một scan từ đường dẫn: `prepare_scan` + `audit_prepared`. Trả `(info, ảnh soát, texture rời)`.
Dùng bởi `audit_scans.py` ở máy local. Notebook gọi thẳng `audit_prepared` với mesh đã nạp sẵn.

In [ ]:
def audit_scan(path: str | Path, target_extent_mm: float = 200.0, override: Optional[dict] = None,
               width: int = 480, height: int = 360, image: bool = True) -> tuple[dict, Optional[np.ndarray], Optional[str]]:
    """Soát một scan: prepare_scan + detect_main_face + áp chỉ định tay (dòng manifest: texture/face/rot90)
    + ảnh soát. Trả (info, ảnh RGB, đường dẫn texture rời hoặc None). info['auto_face'] = kết quả tự động."""
    ov = override or {}
    m = prepare_scan(path, target_extent_mm, ov.get("texture", "auto"))
    info, img = audit_prepared(m, ov, width, height, image)
    return info, img, m.metadata.get("source_texture")

#### `audit_prepared(m_obb, override, width, height, image)`
**Làm gì:** soát mặt khắc cho mesh **đã nạp** (kết quả `prepare_scan`).

**Cách làm:**
1. `detect_main_face` → lưu kết quả tự động vào `info["auto_face"]`.
2. Có chỉ định tay (dòng manifest có `face`) → ghi đè `face`, đặt `confident = True`, `reason` ghi rõ chỉ định tay
   **trùng** hay **KHÁC** kết quả tự động (khác → đáng xem lại).
3. Ghi `rot90`, `has_texture` vào `info`.
4. `image=True` → vẽ ảnh soát bằng `face_audit_image` (cần GPU); `image=False` → trả `None`.

Tách riêng khỏi `audit_scan` để notebook **nạp mỗi scan một lần** (mất 10–20 s với scan 1,4 triệu tam giác) rồi dùng lại.

In [ ]:
def audit_prepared(m_obb: trimesh.Trimesh, override: Optional[dict] = None, width: int = 480, height: int = 360,
                   image: bool = True) -> tuple[dict, Optional[np.ndarray]]:
    """Như audit_scan nhưng nhận mesh ĐÃ nạp (kết quả prepare_scan) -> nạp scan một lần, dùng lại cho mọi bước.
    Trả (info, ảnh RGB hoặc None khi image=False)."""
    ov = override or {}
    info = detect_main_face(m_obb)
    info["auto_face"] = info["face"]
    if ov.get("face", "auto") not in (None, "", "auto"):
        same = "trùng" if ov["face"] == info["auto_face"] else "KHÁC"
        info.update(face=ov["face"], auto=False, confident=True,
                    reason=f"chỉ định qua manifest ({same} kết quả tự động {info['auto_face']})")
    info["rot90"] = int(ov.get("rot90", 0))
    info["has_texture"] = bool(m_obb.metadata.get("has_texture"))
    img = face_audit_image(m_obb, info, width, height) if image else None   # image=False: không cần pyrender/GPU
    return info, img

#### `contact_sheet(files, cols, thumb_w)`
**Làm gì:** ghép nhiều ảnh thành một ảnh lưới để xem nhanh: thu mỗi ảnh về rộng `thumb_w`, đệm đen cho cùng chiều cao,
xếp `cols` cột. Dùng cho ảnh `<tên>_sheet.jpg` của phần A.

In [ ]:
def contact_sheet(files: list[Path], cols: int = 4, thumb_w: int = 320) -> np.ndarray:
    thumbs = []
    for f in files:
        im = cv2.imread(str(f))
        h, w = im.shape[:2]
        thumbs.append(cv2.resize(im, (thumb_w, int(h * thumb_w / w))))
    th = max(t.shape[0] for t in thumbs)
    thumbs = [cv2.copyMakeBorder(t, 0, th - t.shape[0], 0, 0, cv2.BORDER_CONSTANT, value=(0, 0, 0)) for t in thumbs]
    while len(thumbs) % cols:
        thumbs.append(np.zeros_like(thumbs[0]))
    rows = [np.hstack(thumbs[i:i + cols]) for i in range(0, len(thumbs), cols)]
    return np.vstack(rows)

## 2B. Tăng cường hình học (`mocban_enhance/`)
Dùng cho phần B. Không cần GPU.

#### Import & shim
Thư viện của phần B. `scipy.ndimage` cung cấp lọc 1 chiều (`correlate1d`), biến đổi khoảng cách (lấp lỗ) và lọc Gauss.
Lặp lại shim `np.infty` cho trường hợp chạy riêng phần B.

In [ ]:
import math
import cv2
import numpy as np
from scipy.ndimage import distance_transform_edt, gaussian_filter, correlate1d
if not hasattr(np, "infty"):
    np.infty = np.inf

### 1. Chiếu 3D sang 2D (trực giao, góc bất kỳ)
**Bước 3D → 2D của phần B.** Chiếu trực giao mesh vào mặt phẳng ảnh ở góc nhìn bất kỳ, mỗi pixel giữ bề mặt gần camera nhất (Z-buffer) → ảnh độ sâu.

#### `_spherical_dir(elev_deg, azim_deg)`
**Giống hệt** hàm cùng tên ở phần nạp / render: `(cos e·cos a, cos e·sin a, sin e)`. Định nghĩa lại để phần B cũng
dùng được khi tách riêng; hai bản trùng công thức nên định nghĩa sau ghi đè bản trước không đổi gì.

In [ ]:
def _spherical_dir(elev_deg, azim_deg):
    """Hướng đơn vị trên mặt cầu - CÙNG công thức với mocban_render._spherical_dir."""
    e, a = math.radians(elev_deg), math.radians(azim_deg)
    return np.array([math.cos(e) * math.cos(a), math.cos(e) * math.sin(a), math.sin(e)])

#### `view_basis(elev_deg, azim_deg, roll_deg)`
**Làm gì:** hệ trục camera trực giao `(phải, lên, hướng nhìn)` cho góc `elev`, `azim`, `roll`.

Dùng **đúng quy ước** của `look_at_pose` ở phần A: "lên" = trục +Y của khối xoay `roll` quanh Z, trực giao hoá theo
hướng nhìn. Nhờ vậy ảnh phần B cùng hướng với ảnh phần A, và khối luôn đứng ở mọi góc.

`elev = 90°` cho `phải = +X`, `lên = +Y`, `hướng nhìn = −Z` (nhìn thẳng từ trên xuống).

In [ ]:
def view_basis(elev_deg=90.0, azim_deg=0.0, roll_deg=0.0):
    """
    Hệ trục camera trực giao, CÙNG quy ước với mocban_render.look_at_pose: hướng "lên" của ảnh
    là +Y thế giới (trục dọc của khối) xoay roll quanh Z rồi trực giao hoá - nhờ vậy khối luôn
    đứng ở mọi elev/azim, không nhảy hướng ở elev ~ 80 độ.

    Trả (right, up, forward); forward là hướng NHÌN (từ camera vào khối).
    elev=90 cho right=+X, up=+Y, forward=-Z, tức nhìn thẳng từ trên xuống.
    """
    f = -_spherical_dir(elev_deg, azim_deg)          # camera ở ngoài nhìn vào
    r = math.radians(roll_deg)
    up_hint = np.array([-math.sin(r), math.cos(r), 0.0])
    s = np.cross(f, up_hint)
    if np.linalg.norm(s) < 1e-6:                     # nhìn dọc trục +Y -> fallback +Z
        s = np.cross(f, np.array([0.0, 0.0, 1.0]))
    s /= np.linalg.norm(s)
    return s, np.cross(s, f), f

#### `_raster_zbuffer(u, v, dep, F, W, H, max_cand)`
**Làm gì:** Z-buffer bằng cách **rasterize tam giác** — lõi của phép chiếu.

**Đầu vào:** toạ độ pixel liên tục `(u, v)` và độ sâu `dep` của mọi đỉnh, danh sách tam giác `F`, kích thước ảnh.
**Đầu ra:** ảnh độ sâu; ô không có tam giác nào phủ = `−inf`.

**Cách làm:**
- Pixel `(r, c)` có tâm `(c + 0,5; r + 0,5)`. Với mỗi tam giác, thử các tâm pixel nằm trong hộp bao của nó, tính **toạ
  độ barycentric** `(l1, l2, l3)`; tâm nằm trong tam giác khi cả ba ≥ 0.
- Độ sâu tại tâm = nội suy tuyến tính `l1·z1 + l2·z2 + l3·z3`; mỗi pixel giữ giá trị **lớn nhất** = gần camera nhất.
- **Vector hoá:** nhóm tam giác theo cỡ hộp bao (k = 1, 2, 4, 8… tâm pixel mỗi chiều); mỗi nhóm thử k × k tâm cho mọi
  tam giác cùng lúc, chia khối để giới hạn bộ nhớ (`max_cand`).
- Tam giác chiếu thành đoạn thẳng (vách đứng nhìn ngang) không chứa tâm nào → bỏ qua, đúng hình học.

**Vì sao không splat điểm:** splat để lại khe ngẫu nhiên giữa các điểm; ở ô không nhận điểm nào của mặt ngoài, một lớp
bề mặt nằm **dưới** thắng Z-buffer → hố giả sâu 14–35 mm (đo được trên scan mẫu, vì scan có nhiều lớp chồng nhau).
Rasterize tại tâm pixel thì nơi mặt ngoài phủ tâm, lớp dưới không bao giờ thắng. Đo lại: hố giả 871 → 32 pixel, sai số
mặt sin 0,031 → 0,006 mm.

In [ ]:
def _raster_zbuffer(u, v, dep, F, W, H, max_cand=8_000_000):
    """
    Z-buffer bằng RASTERIZE tam giác, đúng tại TÂM pixel: pixel (r, c) có tâm (c + 0.5, r + 0.5) trong toạ độ
    pixel liên tục (u, v) nhận độ sâu, nội suy tuyến tính trên tam giác, của tam giác CAO NHẤT (gần camera nhất)
    chứa tâm đó. Xác định, không ngẫu nhiên: nơi có bề mặt phủ tâm pixel thì lớp bề mặt nằm dưới không bao giờ
    thắng được.

    Vector hoá theo nhóm: tam giác xếp theo số tâm pixel mà hộp bao của nó trải qua (k = 1, 2, 4, 8, ... theo mỗi
    chiều), mỗi nhóm thử k x k tâm ứng viên cho mọi tam giác cùng lúc, chia khối để giới hạn bộ nhớ.
    Tam giác chiếu thành đường thẳng (vách đứng nhìn ngang) không chứa tâm nào -> bỏ qua, đúng hình học.
    """
    buf = np.full(H * W, -np.inf)
    tu, tv, td = u[F], v[F], dep[F]                              # (n, 3)
    c_lo = np.ceil(tu.min(1) - 0.5).astype(np.int64)            # tâm c + 0.5 >= u_min
    c_hi = np.floor(tu.max(1) - 0.5).astype(np.int64)
    r_lo = np.ceil(tv.min(1) - 0.5).astype(np.int64)
    r_hi = np.floor(tv.max(1) - 0.5).astype(np.int64)
    span = np.maximum(c_hi - c_lo, r_hi - r_lo) + 1              # số tâm theo chiều dài hơn của hộp bao
    keep = (c_hi >= c_lo) & (r_hi >= r_lo)                       # hộp bao chứa ít nhất một tâm
    k_cls = np.where(keep, 2 ** np.ceil(np.log2(np.maximum(span, 1))).astype(np.int64), 0)
    for k in np.unique(k_cls[k_cls > 0]):
        idx = np.flatnonzero(k_cls == k)
        oc, orr = np.meshgrid(np.arange(k), np.arange(k))
        oc, orr = oc.ravel(), orr.ravel()
        step = max(1, int(max_cand // (k * k)))
        for s0 in range(0, len(idx), step):
            t = idx[s0:s0 + step]
            cc = c_lo[t, None] + oc[None, :]; rr = r_lo[t, None] + orr[None, :]
            pu, pv = cc + 0.5, rr + 0.5
            au, bu, cu = tu[t, 0, None], tu[t, 1, None], tu[t, 2, None]
            av, bv, cv = tv[t, 0, None], tv[t, 1, None], tv[t, 2, None]
            den = (bv - cv) * (au - cu) + (cu - bu) * (av - cv)
            ok_den = np.abs(den) > 1e-12
            den = np.where(ok_den, den, 1.0)
            l1 = ((bv - cv) * (pu - cu) + (cu - bu) * (pv - cv)) / den
            l2 = ((cv - av) * (pu - cu) + (au - cu) * (pv - cv)) / den
            l3 = 1.0 - l1 - l2
            eps = -1e-9
            ins = (ok_den & (l1 >= eps) & (l2 >= eps) & (l3 >= eps)
                   & (cc <= c_hi[t, None]) & (rr <= r_hi[t, None])
                   & (cc >= 0) & (cc < W) & (rr >= 0) & (rr < H))
            if not ins.any():
                continue
            z = l1 * td[t, 0, None] + l2 * td[t, 1, None] + l3 * td[t, 2, None]
            np.maximum.at(buf, (rr * W + cc)[ins], z[ins])
    return buf.reshape(H, W)

#### `project_depth(mesh, elev_deg, azim_deg, roll_deg, px_mm, out_px, fill_holes)`
**Làm gì:** **phép chiếu 3D → 2D** của phần B: chiếu trực giao mesh vào mặt phẳng ảnh ở góc nhìn bất kỳ, mỗi pixel giữ
bề mặt gần camera nhất.

**Đầu vào:** mesh (mặt khắc hướng +Z), `elev`, `azim`, `roll`, `px_mm` hoặc `out_px`, `fill_holes`.
**Đầu ra:** dict `depth` (mm, 0 = điểm xa nhất, lớn = gần camera), `valid` (pixel có bề mặt), `px_mm`.

**Cách làm:**
1. Hệ trục `view_basis`; mỗi đỉnh: `X = V·phải`, `Y = V·lên`, `D = V·(−hướng nhìn)`.
2. Bước lưới:
   - `px_mm` cho trước → dùng luôn;
   - `out_px` cho trước → cạnh dài ảnh = `out_px` pixel;
   - không có gì → **độ phân giải gốc** của scan: `px = √(diện tích chiếu của phần quay về camera / số đỉnh của phần
     đó)`. Mịn hơn mức này chỉ là nội suy trên tam giác phẳng, không thêm thông tin.
3. Kích thước ảnh `round(phạm vi / px) + 1` (dùng `round`, không `ceil`, để sai số dấu phẩy động không cộng dư một hàng).
4. `_raster_zbuffer` → ảnh độ sâu.
5. `valid` = ô có tam giác phủ, sau một phép đóng 3 × 3 để lấp lỗ kim. `valid = False` còn lại là nền ngoài khối, lỗ
   thủng lớn của scan, hoặc vùng **bị che khuất thật** khi nhìn xiên.
6. `fill_holes`: lấp ô trống bằng giá trị ô có dữ liệu gần nhất (biến đổi khoảng cách), để các bộ lọc phía sau không
   gặp `−inf`.

**Kiểm chứng (cell kiểm chứng giải tích):** mặt sin sai số trung vị 0,006 mm; mặt phẳng nhìn xiên có độ dốc đúng
`px / tan(elev)` (lệch < 0,06 %).

In [ ]:
def project_depth(mesh, elev_deg=90.0, azim_deg=0.0, roll_deg=0.0, px_mm=None, out_px=None, fill_holes=True):
    """
    PHÉP CHIẾU 3D -> 2D: chiếu trực giao mesh vào mặt phẳng ảnh và giữ bề mặt GẦN CAMERA NHẤT trên mỗi pixel
    (Z-buffer), theo góc nhìn bất kỳ.

    Rasterize tam giác (_raster_zbuffer), không splat điểm: độ sâu lấy ĐÚNG TẠI TÂM pixel từ tam giác cao nhất chứa
    tâm đó. Splat điểm (kể cả lấy mẫu ngẫu nhiên trên tam giác) có khe Poisson: ở pixel không nhận mẫu nào của mặt
    trên, một lớp bề mặt nằm DƯỚI (scan thường có lớp trong / mặt chồng) thắng Z-buffer -> hố giả sâu hàng chục mm.
    Rasterize mọi tam giác, KHÔNG lọc theo hướng pháp tuyến: scan hay có tam giác lật pháp tuyến ngay trên mặt ngoài,
    lọc sẽ để lọt lớp dưới; còn mặt sau / lớp trong bị che đúng bởi phép lấy max.

    px_mm: bước lưới ảnh. None (và out_px None) -> độ phân giải GỐC của scan = khoảng cách đỉnh trung bình trên phần
      bề mặt quay về camera; mịn hơn thế chỉ là nội suy trên tam giác phẳng, không thêm thông tin.
    out_px: nếu đặt (và px_mm None) -> px_mm = cạnh dài của ảnh / out_px.

    Trả dict:
      depth [mm] : độ cao so với điểm xa nhất, theo trục nhìn (lớn = gần camera)
      valid      : tâm pixel nằm trên bề mặt (lỗ kim <= 2 px của scan được lấp bằng phép đóng);
                   False = nền ngoài khối, lỗ thủng lớn của scan, hoặc bị CHE KHUẤT thật.
      px_mm      : bước lưới ảnh (đều theo cả hai trục vì mặt phẳng ảnh vuông góc trục nhìn)
    """
    V = np.asarray(mesh.vertices, dtype=np.float64)
    F = np.asarray(mesh.faces)
    s, u, f = view_basis(elev_deg, azim_deg, roll_deg)
    X, Y, D = V @ s, V @ u, V @ (-f)       # D: độ sâu dọc trục nhìn, lớn hơn = gần camera hơn
    if px_mm is None:
        if out_px:
            px_mm = max(X.max() - X.min(), Y.max() - Y.min()) / float(out_px)
        else:
            toward = np.asarray(mesh.face_normals) @ (-f)
            vis = toward > 0.0
            proj_area = float((np.asarray(mesh.area_faces)[vis] * toward[vis]).sum())
            px_mm = math.sqrt(max(proj_area, 1e-12) / max(len(np.unique(F[vis])) if vis.any() else len(V), 1))
    px_mm = float(px_mm)

    # N tâm pixel cách đều bước px_mm phủ (N-1)*px_mm -> round, KHÔNG ceil (ceil cộng dư 1 hàng vì sai số
    # dấu phẩy động).
    W = max(int(round((X.max() - X.min()) / px_mm)) + 1, 2)
    H = max(int(round((Y.max() - Y.min()) / px_mm)) + 1, 2)
    buf = _raster_zbuffer((X - X.min()) / px_mm, (Y.max() - Y) / px_mm, D, F, W, H)   # row 0 ở Y lớn nhất

    hit = np.isfinite(buf)
    k3 = np.ones((3, 3), np.uint8)                # phép đóng: lấp lỗ kim, không nở vùng
    valid = cv2.erode(cv2.dilate(hit.astype(np.uint8), k3), k3).astype(bool)
    if fill_holes and (~hit).any():
        _, (ri, ci) = distance_transform_edt(~hit, return_indices=True)
        buf = buf[ri, ci]
    else:
        buf = np.where(hit, buf, buf[hit].min() if hit.any() else 0.0)
    return {"depth": np.ascontiguousarray((buf - buf.min()).astype(np.float32)),
            "valid": valid, "px_mm": px_mm}

### 2. Chuẩn bị trường độ cao
Ảnh độ sâu của scan cần làm trơn nhẹ trước khi lấy đạo hàm; kèm hàm in các con số chi phối tham số.

#### `prepare_height(h, px_mm, sigma0_px, detrend_sigma_px)`
**Làm gì:** chính quy hoá ảnh độ sâu **trước mọi phép đạo hàm bậc hai**.

**Đầu vào:** `h` (mm), `px_mm`, `sigma0_px` (mặc định 1), `detrend_sigma_px` (tuỳ chọn).

**Vì sao cần:** ảnh độ sâu của scan là mặt ghép từ các **tam giác phẳng** (liên tục nhưng pháp tuyến gãy ở cạnh tam
giác) cộng nhiễu đo của máy scan. Đạo hàm bậc hai của nó là dãy xung trên cạnh tam giác → bản đồ độ cong / MSII hiện lưới
tam giác và hạt nhiễu. Đây là tính chất dữ liệu, không phải lỗi. Làm trơn Gauss σ₀ ≈ 1 px xoá phần này mà gần như không
đụng tới nét khắc (rộng nhiều px).

`detrend_sigma_px`: trừ nền thấp tần `h − G * h` khi khối cong / vênh.

In [ ]:
def prepare_height(h, px_mm, sigma0_px=1.0, detrend_sigma_px=None):
    """
    Chính quy hoá bắt buộc trước mọi đạo hàm bậc hai.

    Ảnh độ sâu của scan là mặt ghép từ các tam giác PHẲNG (liên tục C0 nhưng không C1) cộng nhiễu
    đo của máy scan / photogrammetry ở thang dưới mm. Đạo hàm bậc hai của mặt ghép tam giác là dãy
    xung trên cạnh tam giác -> bản đồ độ cong / MSII hiện lưới tam giác và hạt nhiễu. Đó là tính
    chất của dữ liệu scan, không phải lỗi cài đặt. Làm trơn sigma0 ~ 1 px xoá phần này mà gần như
    không đụng tới nét khắc (rộng nhiều px).

    detrend_sigma_px: nếu đặt, trừ đi nền thấp tần (khối gỗ cong / vênh).
    """
    h = np.ascontiguousarray(np.asarray(h, dtype=np.float32))
    if detrend_sigma_px:
        h = h - gaussian_filter(h, float(detrend_sigma_px), mode="nearest")
    if sigma0_px and sigma0_px > 0:
        h = gaussian_filter(h, float(sigma0_px), mode="nearest")
    return h

#### `relief_stats(h, px_mm)`
**Làm gì:** vài con số chi phối việc chọn tham số: kích thước ảnh, `px_mm`, biên độ độ cao `p99 − p1` (mm và px),
độ cao nhỏ nhất / lớn nhất. In ở đầu phần phân tích.

In [ ]:
def relief_stats(h, px_mm):
    """Vài con số chi phối mọi lựa chọn tham số. In ra đầu notebook."""
    p1, p99 = np.percentile(h, [1, 99])
    relief_mm = float(p99 - p1)
    return {"shape": tuple(h.shape), "px_mm": float(px_mm), "relief_mm": relief_mm,
            "relief_px": relief_mm / float(px_mm),
            "h_min_mm": float(h.min()), "h_max_mm": float(h.max())}

### 3. Đạo hàm & pháp tuyến
Đạo hàm Gauss của ảnh độ sâu (bậc 1 và 2) — nền tảng của pháp tuyến, độ cong, shading.

**Quy ước bắt buộc:** đơn vị mm, lưới bước `px_mm`; `x = cột`, `y = hàng` (frame ảnh); pháp tuyến `n = normalize(-hx, -hy, 1)`. Copy nhầm công thức frame thế giới (Y hướng lên) sẽ lật hướng đèn theo chiều dọc, và nét khắc chìm sẽ trông thành nổi.

#### `_dkernel(sigma_px, order, truncate)`
**Làm gì:** nhân Gauss đạo hàm 1 chiều bậc 0, 1 hoặc 2, **ép chính xác trên đa thức**.

**Cách làm:** lấy mẫu `g(x) = exp(−x²/2σ²)` chuẩn hoá tổng = 1; bậc 1: `(−x/σ²)·g`; bậc 2: `((x² − σ²)/σ⁴)·g`. Rồi ép
hai điều kiện:
- `Σ k = 0` → triệt tiêu thành phần hằng số;
- `Σ k·iⁿ / n! = 1` → đạo hàm đúng tuyệt đối trên đa thức bậc ≤ n.

**Vì sao không dùng `scipy.gaussian_filter(order=2)`:** scipy chỉ chuẩn hoá nhân bậc 0, nên nhân bậc 2 có tổng khác 0
(~1e-4). Phần dư đó nhân với độ cao tuyệt đối rồi rò vào kết quả — toán tử **không bất biến tịnh tiến**. Đo trên bán
cầu R = 46 mm: sai số H 138 %, K 467 % (float64 cũng sai y hệt). Sau khi ép: sai số 0,03 %.

In [ ]:
def _dkernel(sigma_px, order, truncate=4.0):
    """
    Nhân Gauss đạo hàm 1-D, ÉP chính xác trên đa thức.

    KHÔNG dùng thẳng scipy gaussian_filter(order=2): scipy lấy mẫu đạo hàm giải tích của
    Gauss rồi CHỈ chuẩn hoá nhân bậc 0, nên nhân bậc 2 không tổng bằng 0. Phần dư (~1e-4)
    nhân với thành phần một chiều của trường độ cao rồi rò thẳng vào kết quả: đo trên bán cầu
    R = 46 mm sai số H lên tới 138% và K tới 467%. Nói cách khác toán tử đó KHÔNG bất biến
    tịnh tiến, trong khi đạo hàm buộc phải bất biến.

    Ở đây ép hai điều kiện, đúng bằng cấu trúc:
        sum(k) = 0                      -> triệt tiêu hằng số
        sum(k * i^order) / order! = 1   -> đúng chính xác trên đa thức bậc <= order
    """
    r = max(int(truncate * float(sigma_px) + 0.5), int(order) + 1)
    x = np.arange(-r, r + 1, dtype=np.float64)
    sig = float(sigma_px)
    g = np.exp(-x * x / (2.0 * sig * sig))
    g /= g.sum()
    if order == 0:
        return g
    k = (-x / sig ** 2) * g if order == 1 else ((x * x - sig * sig) / sig ** 4) * g
    k = k - k.mean()                                        # sum(k) = 0
    k = k / ((k * x ** order).sum() / math.factorial(order))  # đúng trên x^order
    return k

#### `_sepfilt(h, sigma_px, order_row, order_col)`
**Làm gì:** lọc tách được: nhân bậc `order_row` theo trục hàng rồi nhân bậc `order_col` theo trục cột. Ví dụ `(0, 1)` =
`∂/∂x` (làm trơn theo hàng, đạo hàm theo cột). Biên xử lý kiểu `nearest`.

In [ ]:
def _sepfilt(h, sigma_px, order_row, order_col):
    """Lọc tách được: bậc order_row dọc trục 0 (row), bậc order_col dọc trục 1 (col)."""
    out = correlate1d(h, _dkernel(sigma_px, order_row), axis=0, mode="nearest")
    return correlate1d(out, _dkernel(sigma_px, order_col), axis=1, mode="nearest")

#### `derivatives(h, px_mm, sigma_px)`
**Làm gì:** năm đạo hàm Gauss của ảnh độ sâu: `hx, hy` (vô thứ nguyên) và `hxx, hxy, hyy` (đơn vị 1/mm).

**Quy ước:** `x` = cột, `y` = hàng. Nhân lọc **không** chia bước lưới nên phải chia `px_mm` (bậc 1) hoặc `px_mm²`
(bậc 2) bằng tay. Không trộn với `np.gradient` (hàm đó **có** chia bước lưới) — trộn hai kiểu là nguồn lỗi đơn vị hay
gặp nhất.

In [ ]:
def derivatives(h, px_mm, sigma_px=1.0):
    """
    Đạo hàm Gauss của trường độ cao. Nhân KHÔNG chia bước lưới nên phải chia px_mm bằng tay.

    Trả hx, hy (vô thứ nguyên) và hxx, hxy, hyy (1/mm), với x = col, y = row.
    """
    h = np.asarray(h, dtype=np.float64)
    s = float(px_mm)
    sig = float(sigma_px) if sigma_px and sigma_px > 0 else 1e-6
    f = lambda a, b: _sepfilt(h, sig, a, b).astype(np.float32)
    return {"hx": f(0, 1) / s, "hy": f(1, 0) / s,
            "hxx": f(0, 2) / (s * s), "hyy": f(2, 0) / (s * s), "hxy": f(1, 1) / (s * s)}

#### `normals(h, px_mm, sigma_px, gain)`
**Làm gì:** pháp tuyến đơn vị trong frame ảnh: `n = normalize(−g·hx, −g·hy, 1)`.

**`gain` (g) < 1 nén độ dốc:** mép nét khắc dốc tới ~2,4 nên pháp tuyến thô cho nền một màu cộng viền 2 px cháy sáng.
`gain = 0,35` giữ được cả nền lẫn mép.

In [ ]:
def normals(h, px_mm, sigma_px=1.0, gain=1.0):
    """
    Pháp tuyến đơn vị trong FRAME ẢNH: n = normalize(-gain*hx, -gain*hy, 1).

    gain < 1 nén độ dốc. Cần thiết ở đây: nét khắc cao 1.5 mm với mép chỉ ~1.6 px nên độ dốc
    nhảy 0 -> ~2.4, normal map thô sẽ ra nền phẳng một màu cộng viền 2 px cháy sáng.
    """
    d = derivatives(h, px_mm, sigma_px)
    g = float(gain)
    n = np.stack([-g * d["hx"], -g * d["hy"], np.ones_like(d["hx"])], axis=-1)
    return (n / np.linalg.norm(n, axis=-1, keepdims=True)).astype(np.float32)

#### `normal_map_rgb(n)`
**Làm gì:** pháp tuyến → ảnh màu theo quy ước chuẩn `RGB = (n + 1)/2 · 255`. Không kéo giãn percentile, để màu có
nghĩa tuyệt đối (xanh tím = phẳng hướng lên).

In [ ]:
def normal_map_rgb(n):
    """Pháp tuyến -> RGB uint8 theo quy ước (n+1)/2. Không kéo giãn percentile."""
    return np.clip((n + 1.0) * 0.5 * 255.0, 0, 255).astype(np.uint8)

#### `slope_aspect_hsv(h, px_mm, sigma_px)`
**Làm gì:** mã hoá độ dốc bằng màu HSV: **màu (hue) = hướng dốc**, **độ sáng = độ lớn dốc** (`tanh(|∇h| / p95)`).

Trên nét khắc thường đọc rõ hơn normal map RGB, vì mắt phân tách nét theo hướng dốc: hai vách đối diện của một nét có
màu đối nhau.

In [ ]:
def slope_aspect_hsv(h, px_mm, sigma_px=1.0):
    """
    Độ dốc/hướng dốc mã hoá HSV: hue = hướng dốc, value = độ lớn dốc.
    Trên nét khắc CJK đọc rõ hơn hẳn normal map RGB vì mắt phân tách nét theo hướng.
    """
    d = derivatives(h, px_mm, sigma_px)
    mag = np.hypot(d["hx"], d["hy"])
    ref = float(np.percentile(mag, 95)) or 1.0
    hue = (np.degrees(np.arctan2(d["hy"], d["hx"])) % 360.0) / 2.0        # OpenCV: 0..179
    val = np.tanh(mag / ref)
    hsv = np.stack([hue, np.full_like(hue, 255.0), val * 255.0], -1).astype(np.uint8)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

### 4. Bản đồ độ sâu
**Phương pháp 1** của phần B.

#### `depth_map(h, px_mm, pct, hp_sigma_px)`
**Làm gì:** ba dạng của cùng ảnh độ sâu:
- `mm`: độ cao thô, đơn vị vật lý;
- `stretch`: kéo giãn theo percentile 1–99 (bỏ viền 8 px), về [0, 1];
- `local = h − G_σ * h` (σ mặc định 12 px): **khử nền thấp tần**.

Bản `local` mới lộ nét khắc khi khối cong / vênh / nứt, nên là bản đưa vào lưới so sánh (**phương pháp 1**).

In [ ]:
def depth_map(h, px_mm, pct=(1, 99), hp_sigma_px=12.0):
    """
    Ba dạng của cùng một trường độ cao:
      mm      : thô, đơn vị vật lý (panel duy nhất có colorbar mm thật)
      stretch : kéo giãn percentile robust, [0,1]
      local   : h - G_sigma * h, khử nền thấp tần. Đây mới là bản lộ nét khắc khi tấm
                cong hoặc nứt, nên dùng bản này trong lưới so sánh.
    """
    h = np.asarray(h, dtype=np.float32)
    lo, hi = _interior_percentile(h, pct, margin_px=8)
    stretch = np.clip((h - lo) / max(hi - lo, 1e-9), 0, 1)
    local = h - gaussian_filter(h, float(hp_sigma_px), mode="nearest")
    return {"mm": h, "stretch": stretch.astype(np.float32), "local": local.astype(np.float32),
            "lo_mm": float(lo), "hi_mm": float(hi)}

### 5. Độ cong (Monge patch chính xác)
**Phương pháp 3** của phần B.

#### `curvature(h, px_mm, sigma_px)`
**Làm gì:** độ cong **chính xác** của mặt Monge `z = h(x, y)` (**phương pháp 3**).

Với `p = hx, q = hy, r = hxx, s = hxy, t = hyy`, `W = √(1 + p² + q²)`:
- độ cong Gauss `K = (rt − s²) / W⁴` (1/mm²);
- độ cong trung bình `H = ((1+q²)r − 2pqs + (1+p²)t) / (2W³)` (1/mm);
- độ cong chính `k1, k2 = H ± √(H² − K)`;
- shape index `S = (2/π)·atan(H / √(H² − K))` ∈ [−1, 1] (hình dạng: lõm ↔ yên ngựa ↔ lồi), curvedness
  `C = √(2H² − K)` (độ mạnh).

**Dấu:** pháp tuyến hướng lên → phần **nổi** có `H < 0`.

**Vì sao không dùng `H ≈ ½∇²h`:** xấp xỉ chỉ đúng khi độ dốc ≪ 1; mép nét khắc dốc hơn nhiều. Công thức chính xác chỉ
tốn thêm 3 phép lọc. `max(H² − K, 0)` tránh NaN do float32 làm hiệu này âm nhẹ ở vùng phẳng.

In [ ]:
def curvature(h, px_mm, sigma_px=1.5):
    """
    Độ cong CHÍNH XÁC cho mặt Monge z = h(x,y). KHÔNG dùng xấp xỉ Laplacian.

        W = sqrt(1 + p^2 + q^2),   p = hx, q = hy, r = hxx, s = hxy, t = hyy
        K = (r*t - s^2) / W^4                                   [1/mm^2]
        H = ((1+q^2)*r - 2*p*q*s + (1+p^2)*t) / (2 * W^3)       [1/mm]
        k1, k2 = H +- sqrt(max(H^2 - K, 0))

    Vì sao không dùng Laplacian: H ~ (1/2)*lap(h) chỉ đúng khi độ dốc << 1. Ở đây độ dốc
    mép nét đạt ~2.4 nên W^3 ~ 17.6 (còn ~5.9 sau khi làm trơn 1 px). Laplacian thổi phồng
    |H| 6-18 lần ĐÚNG NGAY MÉP NÉT - nơi chứa toàn bộ tín hiệu - rồi bước chuẩn hoá
    percentile sẽ nghiền phần còn lại thành xám. Công thức đúng chỉ tốn thêm 3 convolution.

    DẤU: với n = (-p,-q,1)/W hướng lên, nét NỔI là cực đại địa phương nên H < 0.

    shape_index  S = (2/pi)*atan(H / sqrt(H^2 - K))  thuộc [-1,1], bất biến tỉ lệ.
    curvedness   C = sqrt(2*H^2 - K)                 độ lớn không dấu [1/mm].
    """
    d = derivatives(h, px_mm, sigma_px)
    p, q = d["hx"], d["hy"]
    r, s, t = d["hxx"], d["hxy"], d["hyy"]

    W2 = 1.0 + p * p + q * q
    W = np.sqrt(W2)
    K = (r * t - s * s) / (W2 * W2)
    H = ((1.0 + q * q) * r - 2.0 * p * q * s + (1.0 + p * p) * t) / (2.0 * W2 * W)

    disc = np.sqrt(np.maximum(H * H - K, 0.0))  # float32 làm H^2-K âm nhẹ ở vùng phẳng -> NaN
    k1, k2 = H + disc, H - disc
    shape_index = (2.0 / math.pi) * np.arctan(H / (disc + 1e-12))
    curvedness = np.sqrt(np.maximum(2.0 * H * H - K, 0.0))
    return {"H": H.astype(np.float32), "K": K.astype(np.float32),
            "k1": k1.astype(np.float32), "k2": k2.astype(np.float32),
            "shape_index": shape_index.astype(np.float32),
            "curvedness": curvedness.astype(np.float32)}

#### `normal_curvature_dir(h, px_mm, ldir, sigma_px)`
**Làm gì:** độ cong pháp tuyến **dọc phương vị của đèn**:
`k_l = (ux²·hxx + 2·ux·uy·hxy + uy²·hyy) / W³`, với `u` = hướng đèn chiếu xuống mặt phẳng.

Là thành phần "versatile" của Radiance Scaling: gờ **hướng về phía đèn** được làm sáng lên.

In [ ]:
def normal_curvature_dir(h, px_mm, ldir, sigma_px=1.5):
    """
    Độ cong pháp tuyến dọc phương vị của đèn - thành phần "versatile" của Radiance Scaling.
    u = normalize(ldir_xy);  k_l = (ux^2*hxx + 2*ux*uy*hxy + uy^2*hyy) / W^3
    """
    d = derivatives(h, px_mm, sigma_px)
    u = np.asarray(ldir, dtype=np.float64)[:2]
    nrm = float(np.linalg.norm(u))
    ux, uy = (u / nrm) if nrm > 1e-9 else (1.0, 0.0)
    W2 = 1.0 + d["hx"] ** 2 + d["hy"] ** 2
    return ((ux * ux * d["hxx"] + 2.0 * ux * uy * d["hxy"] + uy * uy * d["hyy"])
            / (W2 * np.sqrt(W2))).astype(np.float32)

### 6. Bất biến tích phân đa tỉ lệ (MSII)
**Phương pháp 4** của phần B.

#### `_disk_kernel(r_px)`
**Làm gì:** nhân lọc hình đĩa bán kính `r` px, tổng = 1 → phép lọc trả **trung bình trong đĩa**. Dùng cho MSII.

In [ ]:
def _disk_kernel(r_px):
    r = max(int(round(r_px)), 1)
    y, x = np.mgrid[-r:r + 1, -r:r + 1]
    k = ((x * x + y * y) <= r * r).astype(np.float32)
    return k / float(k.sum())

#### `msii_volume(h, px_mm, radii_px, exact, n_rho, n_phi)`
**Làm gì:** bất biến tích phân đa tỉ lệ MSII (Mara & Krömker) — **phương pháp 4**.

**Định nghĩa:** `v_r(p)` = tỉ lệ thể tích của khối vật liệu nằm trong quả cầu bán kính `r` tâm tại điểm bề mặt `p`.
Mặt phẳng → đúng ½; điểm lồi (nổi) → < ½; điểm lõm → > ½.

**Liên hệ độ cong** (khai triển Pottmann): `v_r − ½ ≈ (3/16)·H·r` — tức MSII là độ cong trung bình **có tham số tỉ
lệ** `r`, nhìn được chi tiết ở nhiều cỡ.

**Tính trên ảnh độ sâu:**
- mặc định (xấp xỉ): `v_r ≈ ½ + 3/(4r)·(trung bình đĩa r của h − h)` → **một phép lọc đĩa** cho mỗi bán kính;
- `exact=True`: cầu phương cực `n_rho × n_phi` mẫu, cắt độ cao theo chiều cao chỏm cầu `√(r² − ρ²)` — chậm hơn nhưng
  đúng ở mọi `r`.

**Giới hạn của xấp xỉ:** chỉ đúng khi `r ≳ 2 × độ sâu nét`; bán kính nhỏ hơn thì **bão hoà** (cell MSII in cảnh báo).
Kết quả luôn kẹp [0, 1] như tích phân thật.

**So với DoG:** cùng họ, nhưng MSII có mốc ½, chuẩn hoá 1/r và bị chặn — không "nổ" ở vách đứng.

In [ ]:
def msii_volume(h, px_mm, radii_px=(3, 6, 12, 24, 48), exact=False, n_rho=6, n_phi=12):
    """
    Bất biến tích phân đa tỉ lệ (Mara & Kroemker), bản trường độ cao.

    Định nghĩa gốc: v_r(p) = Vol(B_r(p) giao Solid) / ((4/3)*pi*r^3). Mặt phẳng cho đúng
    v_r = 1/2. Khai triển Pottmann Vol = (2*pi/3)r^3 - (pi/4)*H_out*r^4 + O(r^5) với H_out
    lấy theo pháp tuyến NGOÀI của khối (lồi -> dương), quy về quy ước Monge của module này
    (H âm trên phần nổi, tức H = -H_out):

        v_r - 1/2 = (3/16) * H * r + O(r^2)            [đã kiểm chứng số: sai lệch 0.5%]

    tức v_r là ước lượng độ cong trung bình có tham số tỉ lệ. Đây là lý do DoG "cùng họ" -
    nhưng DoG thiếu mốc 1/2, thiếu chuẩn hoá 1/r và không bị chặn nên nổ ở vách đứng rồi
    nuốt hết dải percentile.

    Với trường độ cao, tích phân là chính xác:
        v_r = 1/2 + 3/(4*pi*r^3) * TichPhan_{rho<=r} clip(d, -c(rho), +c(rho)) dA
    với d = h(x,y) - h(p) và c(rho) = sqrt(r^2 - rho^2) là nửa chiều cao cột cầu.

    exact=False (mặc định): bỏ clip -> tích phân thành MỘT convolution đĩa:
        v_r ~ 1/2 + 3/(4*r_mm) * (trung_binh_dia_r(h) - h)
    Clip chỉ bất hoạt khi r_mm >~ 2*relief_mm (~7.8 px ở cấu hình chuẩn), nên r <= 6 px bão
    hoà và chỉ dùng định tính. Luôn kẹp [0,1] vì tích phân thật bảo đảm điều đó.

    exact=True: cầu phương cực n_rho x n_phi mẫu, có clip. Đắt hơn ~n lần nhưng đúng ở mọi r.

    Trả mảng HxWxR trong [0,1].
    """
    h = np.ascontiguousarray(np.asarray(h, dtype=np.float32))
    out = np.empty(h.shape + (len(radii_px),), dtype=np.float32)

    for j, r_px in enumerate(radii_px):
        r_mm = float(r_px) * float(px_mm)
        if not exact:
            mean_r = cv2.filter2D(h, -1, _disk_kernel(r_px), borderType=cv2.BORDER_REPLICATE)
            v = 0.5 + (3.0 / (4.0 * r_mm)) * (mean_r - h)
        else:
            acc = np.zeros_like(h)
            base_y, base_x = _base_maps(h.shape)
            mapx, mapy = np.empty_like(base_x), np.empty_like(base_y)
            d_rho, d_phi = r_px / n_rho, 2.0 * math.pi / n_phi
            for i in range(n_rho):
                rho_px = (i + 0.5) * d_rho
                c_mm = math.sqrt(max(r_mm * r_mm - (rho_px * px_mm) ** 2, 0.0))
                w = rho_px * d_rho * d_phi * (px_mm ** 2)   # rho*drho*dphi, đổi sang mm^2
                for m in range(n_phi):
                    phi = m * d_phi
                    hs = _shift(h, rho_px * math.cos(phi), rho_px * math.sin(phi),
                                mapx, mapy, base_x, base_y)
                    acc += np.clip(hs - h, -c_mm, c_mm) * w
            v = 0.5 + acc * (3.0 / (4.0 * math.pi * r_mm ** 3))
        out[..., j] = np.clip(v, 0.0, 1.0)
    return out

#### `msii_combine(V, radii_px, px_mm)`
**Làm gì:** gộp MSII nhiều bán kính:
- `combined = Σ (v_j − ½) / r_j` → bản đồ đa tỉ lệ đưa vào lưới so sánh;
- `bands = v_j − v_{j+1}` → tách đặc trưng theo dải tỉ lệ;
- `scale_argmax` = bán kính có `|v − ½|` lớn nhất → "tỉ lệ đặc trưng" của từng điểm.

In [ ]:
def msii_combine(V, radii_px, px_mm):
    """
    combined    : tổng (v_j - 1/2)/r_j  -> bản đồ đa tỉ lệ đưa vào lưới so sánh
    bands       : v_j - v_{j+1}, tách đặc trưng theo dải tỉ lệ
    scale_argmax: tỉ lệ có |v - 1/2| lớn nhất -> bản đồ "tỉ lệ đặc trưng" (colormap phân loại)
    """
    dev = V - 0.5
    r_mm = np.asarray(radii_px, dtype=np.float32) * float(px_mm)
    combined = (dev / r_mm[None, None, :]).sum(axis=-1)
    bands = dev[..., :-1] - dev[..., 1:]
    return {"combined": combined.astype(np.float32), "bands": bands.astype(np.float32),
            "scale_argmax": np.argmax(np.abs(dev), axis=-1).astype(np.int32)}

### 7. Ambient Occlusion (horizon mapping)
**Phương pháp 5** của phần B.

#### `_base_maps(shape)`
**Làm gì:** lưới toạ độ pixel `(hàng, cột)` dạng float32 — đầu vào cho `cv2.remap` khi dịch ảnh.

In [ ]:
def _base_maps(shape):
    H, W = shape
    base_y, base_x = np.mgrid[0:H, 0:W]
    return base_y.astype(np.float32), base_x.astype(np.float32)

#### `_shift(h, dx_px, dy_px, mapx, mapy, base_x, base_y)`
**Làm gì:** lấy mẫu ảnh `h` tại vị trí lệch `(dx, dy)` pixel (có thể lẻ) bằng `cv2.remap` nội suy song tuyến; ngoài
biên lặp giá trị biên. Dùng chung cho AO và MSII chính xác.

In [ ]:
def _shift(h, dx_px, dy_px, mapx, mapy, base_x, base_y):
    """Lấy mẫu h tại offset (dx,dy) px. BORDER_REPLICATE, KHÔNG clip toạ độ."""
    np.add(base_x, np.float32(dx_px), out=mapx)
    np.add(base_y, np.float32(dy_px), out=mapy)
    return cv2.remap(h, mapx, mapy, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)

#### `horizon_tan(h, px_mm, az_rad, radius_px, n_steps, log_steps)`
**Làm gì:** tan của **góc chân trời** theo một phương vị:
`tan θ_h = max_{t ∈ (0, R]} (h(p + t·u) − h(p)) / t`, kẹp ≥ 0 (t tính bằng mm).

Tức là: đứng ở điểm `p`, nhìn theo hướng `u`, bề mặt xung quanh che khuất tới góc nào.

Bước `t` chia theo thang **log** (12 bước từ 1 tới `R` px): tìm chân trời không phụ thuộc tỉ lệ, 12 bước log thay được
24 bước đều.

In [ ]:
def horizon_tan(h, px_mm, az_rad, radius_px=24, n_steps=12, log_steps=True):
    """
    tan của góc chân trời dọc một phương vị:
        tan(theta_h) = max_{t in (0,R]} ( h(p + t*u) - h(p) ) / t,   t tính bằng mm, kẹp >= 0.
    Bước log vì phát hiện chân trời vô hướng tỉ lệ -> 12 bước log đủ thay 24 bước đều.
    """
    h = np.ascontiguousarray(np.asarray(h, dtype=np.float32))
    base_y, base_x = _base_maps(h.shape)
    mapx, mapy = np.empty_like(base_x), np.empty_like(base_y)
    ca, sa = math.cos(az_rad), math.sin(az_rad)

    if log_steps:
        ks = np.unique(np.round(np.geomspace(1.0, float(radius_px), n_steps)))
    else:
        ks = np.linspace(1.0, float(radius_px), n_steps)

    best = np.zeros_like(h)
    for k in ks:
        hs = _shift(h, k * ca, k * sa, mapx, mapy, base_x, base_y)
        np.maximum(best, (hs - h) / (float(k) * float(px_mm)), out=best)
    return best

#### `horizon_ao(h, px_mm, n_az, radius_px, n_steps, log_steps)`
**Làm gì:** Ambient Occlusion — mức độ một điểm "hở" ra bầu trời (**phương pháp 5**). 1 = hở hoàn toàn, 0 = bị che kín.

**Công thức:** tích phân bán cầu có trọng số cosine; với mỗi phương vị phần nhìn thấy là `[0, π/2 − θ_h]` và
`∫cos ψ sin ψ dψ = ½cos²θ_h`, nên
`AO = trung bình theo phương vị [cos²θ_h] = trung bình [1 / (1 + tan²θ_h)]` — **không cần arctan**.

**Kiểm chứng:** mặt phẳng → AO = 1; chân tường đứng → AO ≈ 0,5.

Tham số: `n_az` phương vị (16), `radius_px` bán kính tìm chân trời (24) — bán kính quyết định cỡ chi tiết được làm nổi.

In [ ]:
def horizon_ao(h, px_mm, n_az=16, radius_px=24, n_steps=12, log_steps=True):
    """
    Ambient occlusion theo chân trời, tích phân bán cầu có trọng số cosine.

    Với mỗi phương vị, dải cực nhìn thấy là [0, pi/2 - theta_h] và
    tích_phan cos(psi)sin(psi) dpsi = (1/2)cos^2(theta_h), nên

        AO = trung_binh_phi [ cos^2(theta_h) ] = trung_binh_phi [ 1 / (1 + tan^2 theta_h) ]

    Không cần arctan. Kiểm chứng: mặt phẳng -> AO = 1; chân tường đứng -> AO = 0.5.
    1 = hở hoàn toàn, 0 = bị che hoàn toàn.

    Nâng cấp khả dĩ (không cài ở đây): GTAO (Jimenez 2016) có trọng số max(0, n.omega) cho
    mặt tiếp tuyến nghiêng; ở dữ liệu này chỉ đổi phần vách nét.
    """
    acc = None
    for i in range(n_az):
        tan_h = horizon_tan(h, px_mm, 2.0 * math.pi * i / n_az, radius_px, n_steps, log_steps)
        term = 1.0 / (1.0 + tan_h * tan_h)
        acc = term if acc is None else acc + term
    return (acc / float(n_az)).astype(np.float32)

### 8. Exaggerated Shading & Radiance Scaling
**Phương pháp 6** của phần B, kèm Lambert làm đối chứng.

#### `light_dir(elev_deg, azim_deg)`
**Làm gì:** vector hướng đèn trong **frame ảnh** (x = cột, y = hàng, z hướng ra khỏi mặt) từ góc cao và phương vị.
Mặc định 35° / 135° (đèn từ góc trên trái — quy ước quen mắt cho ảnh nổi).

In [ ]:
def light_dir(elev_deg=35.0, azim_deg=135.0):
    """Hướng đèn đơn vị trong frame ảnh (x = col, y = row, z hướng ra ngoài mặt)."""
    e, a = math.radians(elev_deg), math.radians(azim_deg)
    return np.array([math.cos(e) * math.cos(a), math.cos(e) * math.sin(a), math.sin(e)],
                    dtype=np.float32)

#### `lambert(h, px_mm, ldir, sigma_px, gain)`
**Làm gì:** tô bóng Lambert thuần `max(0, n · l)` — **đối chứng**. Không có nó thì không chứng minh được exaggerated
shading / radiance scaling tốt hơn đèn thường.

In [ ]:
def lambert(h, px_mm, ldir, sigma_px=1.0, gain=1.0):
    """Lambert thuần - ĐỐI CHỨNG. Không có nó thì không chứng minh được 6a/6b thắng."""
    n = normals(h, px_mm, sigma_px, gain)
    return np.clip((n * np.asarray(ldir, np.float32)[None, None, :]).sum(-1), 0, 1).astype(np.float32)

#### `exaggerated_shading(h, px_mm, ldir, sigma0_px, n_scales, gain, contrast)`
**Làm gì:** Exaggerated Shading (theo tinh thần Rusinkiewicz et al. 2006) — **phương pháp 6a**.

**Cách làm:**
1. Làm trơn ảnh độ sâu ở nhiều tỉ lệ `σ_i = σ₀·2^i` (i = 0…4), tính pháp tuyến `n_i` và bóng `s_i = l · n_i`.
2. Chi tiết của tỉ lệ `i` = phần bóng mà tỉ lệ đó thêm vào so với tỉ lệ thô hơn:
   `S_i = clip(½ + c·(s_i − s_{i+1}), 0, 1)` (c = 6).
3. Gộp nhân qua các tỉ lệ: `S = (Π S_i)^(1/L)`.

**Vì sao hiệu `s_i − s_{i+1}`:** bỏ thành phần bóng thô — vốn phụ thuộc hướng đèn và làm mất chi tiết khi đèn thấp — chỉ
giữ phần nổi / chìm ở từng tỉ lệ. Kết quả gần như **không đổi theo góc đèn** (xem so sánh với Lambert ở cell phương
pháp 6).

Bản trước dùng tỉ số thay cho hiệu và dồn mọi giá trị về ~0,5 (ảnh xám phẳng); dạng hiệu cho dải gần kín [0, 1].

In [ ]:
def exaggerated_shading(h, px_mm, ldir, sigma0_px=1.0, n_scales=4, gain=0.4, contrast=6.0):
    """
    Exaggerated Shading theo tinh thần Rusinkiewicz et al. 2006 (không phải port nguyên bản).

    Cơ chế gốc: dựng kim tự tháp pháp tuyến từ hình học làm trơn dần; ở mỗi tỉ lệ đèn được
    tham chiếu lại về gần phương tiếp tuyến của tỉ lệ THÔ HƠN, nên cái được tô bóng là phần
    CHI TIẾT mà tỉ lệ đó thêm vào so với tỉ lệ thô hơn, rồi gộp nhân qua các tỉ lệ:

        sigma_i = sigma0 * 2^i;   n_i từ G_{sigma_i} * h;   s_i = l . n_i
        S_i = clip( 0.5 + contrast * (s_i - s_{i+1}), 0, 1 )     <- chi tiết của tỉ lệ i
        S   = ( tich_i S_i ) ^ (1/L)

    Lấy HIỆU giữa hai tỉ lệ liên tiếp chính là hiện thực của bước tham chiếu lại đèn: nó bỏ
    đi thành phần bóng thô (vốn phụ thuộc hướng đèn và làm mất chi tiết ở góc chiếu thấp) và
    giữ lại đúng phần nổi/chìm ở tỉ lệ đang xét.

    LƯU Ý: bản trước dùng tỉ số S_i/(G*S_i) làm chuẩn hoá tương phản; nó dồn mọi giá trị về
    ~0.5 (đo được dải [0.44, 0.53] trên tấm thật) nên panel ra xám phẳng. Dạng hiệu + hệ số
    contrast cho dải gần kín [0,1].
    """
    ldir = np.asarray(ldir, dtype=np.float32)
    h32 = np.asarray(h, dtype=np.float32)
    shades = []
    for i in range(int(n_scales) + 1):
        sig = float(sigma0_px) * (2.0 ** i)
        n_i = normals(gaussian_filter(h32, sig, mode="nearest"), px_mm,
                      sigma_px=max(sig, 1e-6), gain=gain)
        shades.append((n_i * ldir[None, None, :]).sum(-1))

    prod = np.ones_like(h32)
    for i in range(int(n_scales)):
        prod *= np.clip(0.5 + float(contrast) * (shades[i] - shades[i + 1]), 0, 1)
    return np.power(np.maximum(prod, 0.0), 1.0 / float(n_scales)).astype(np.float32)

#### `radiance_scaling(h, px_mm, ldir, alpha, sigma_px, kappa, kref_pct, gain)`
**Làm gì:** Radiance Scaling (Vergne et al. 2010) — **phương pháp 6b**: `L' = σ(κ̄) · L_lambert`.

**Hàm scaling** (hàm Möbius) xác định bởi 3 tính chất: `σ(0) = 1`, `σ(+1) = α` (lồi sáng lên α lần),
`σ(−1) = 1/α` (lõm tối đi α lần):
`σ(κ̄) = ((α+1) + (α−1)κ̄) / ((α+1) − (α−1)κ̄)`.

**Chuẩn hoá độ cong bắt buộc:** `κ̄ = (2/π)·atan(κ / κ_ref)` với `κ_ref` = percentile 95 của `|κ|` — độ cong thô không bị
chặn; tự chuẩn theo percentile làm `α` thành núm vặn ổn định giữa các khối khác nhau.

`kappa="directional"` dùng `normal_curvature_dir` (gờ hướng về đèn sáng lên); `"mean"` dùng độ cong trung bình H.

In [ ]:
def radiance_scaling(h, px_mm, ldir, alpha=3.0, sigma_px=1.5, kappa="directional",
                     kref_pct=95, gain=1.0):
    """
    Radiance Scaling (Vergne et al., I3D 2010):  L'(p) = sigma(kappa_bar(p)) * L(p).

    Hàm scaling là hàm hữu tỉ (Moebius) xác định bởi ba tính chất:
        sigma(0) = 1,   sigma(+1) = alpha,   sigma(-1) = 1/alpha,   đơn điệu ở giữa
    tức lồi được làm sáng lên alpha lần, lõm bị tối đi alpha lần (đối xứng trên thang log):

        sigma(kb) = ((alpha+1) + (alpha-1)*kb) / ((alpha+1) - (alpha-1)*kb)

    (Ba tính chất trên là phần quyết định hành vi; đại số nguyên văn trong bài báo chưa được
    đối chiếu ở đây, nên markdown nên trích ba tính chất thay vì khẳng định công thức gốc.)

    Ánh xạ độ cong là BẮT BUỘC vì kappa thô không bị chặn:
        kappa_bar = (2/pi) * atan(kappa / kappa_ref),  kappa_ref = percentile(|kappa|, 95)
    Tự chuẩn theo percentile làm alpha thành núm ổn định giữa các tấm khác nhau.

    kappa="directional" dùng độ cong pháp tuyến dọc phương vị đèn (tạo hiệu ứng đặc trưng
    "gờ hướng về đèn sáng lên"); "mean" dùng độ cong trung bình H.
    """
    k = (normal_curvature_dir(h, px_mm, ldir, sigma_px) if kappa == "directional"
         else curvature(h, px_mm, sigma_px)["H"])
    kref = float(np.percentile(np.abs(_interior(k, 8)), kref_pct)) or 1.0
    kb = (2.0 / math.pi) * np.arctan(k / kref)
    a = float(alpha)
    sig = ((a + 1.0) + (a - 1.0) * kb) / ((a + 1.0) - (a - 1.0) * kb)
    return np.clip(sig * lambert(h, px_mm, ldir, sigma_px, gain), 0, 1).astype(np.float32)

### 9. Chuẩn hoá hiển thị
Đưa kết quả về ảnh hiển thị theo **một chính sách duy nhất**. File `s09_display.py`.

#### `_interior(x, margin_px)`
**Làm gì:** cắt bỏ viền `margin_px` pixel mỗi cạnh. Viền ảnh độ sâu thường là mép khối dốc đứng, nếu để lại sẽ chi phối
các percentile dùng để chuẩn hoá.

In [ ]:
def _interior(x, margin_px=8):
    m = int(margin_px)
    return x[m:-m, m:-m] if (m > 0 and x.shape[0] > 2 * m and x.shape[1] > 2 * m) else x

#### `_interior_percentile(x, pct, margin_px)`
**Làm gì:** hai percentile `(thấp, cao)` tính trên phần **bên trong** (đã bỏ viền) của ảnh.

In [ ]:
def _interior_percentile(x, pct, margin_px=8):
    xi = _interior(np.asarray(x, dtype=np.float32), margin_px)
    return (float(np.percentile(xi, pct[0])), float(np.percentile(xi, pct[1])))

#### `norm01(x, sym, pct, margin_px, vrange)`
**Làm gì:** đưa một đại lượng về [0, 1] để hiển thị, theo **một chính sách duy nhất** cho cả notebook.

| Tuỳ chọn | Thang |
|---|---|
| mặc định | percentile 2–98 phần bên trong |
| `sym=True` | đối xứng quanh 0: `[−a, a]` với `a` = percentile 98 của `|x|` → 0 luôn nằm giữa (dùng cho đại lượng có dấu) |
| `vrange=(lo, hi)` | khoá thang tay — **bắt buộc khi quét tham số**, nếu không mỗi ảnh tự chuẩn hoá và dãy so sánh vô nghĩa |

**Không** dùng cho đại lượng đã bị chặn sẵn (AO, MSII `v_r`, mọi shading): kéo giãn sẽ phá mốc "½ = phẳng",
"1 = hở", vốn là toàn bộ ý nghĩa của chúng.

In [ ]:
def norm01(x, sym=False, pct=(2, 98), margin_px=8, vrange=None):
    """
    Chuẩn hoá về [0,1] theo một chính sách duy nhất cho cả notebook.

    sym=True: đối xứng quanh 0, thang a = percentile(|x|, pct[1]) -> 0 luôn rơi vào 0.5.
    vrange=(lo,hi): khoá thang thủ công. BẮT BUỘC dùng khi sweep tham số, nếu không mỗi panel
    tự chuẩn hoá và cả dãy sweep chẳng cho thấy gì.

    KHÔNG gọi hàm này cho đại lượng đã bị chặn sẵn (AO, MSII v_r, mọi shading): kéo giãn
    chúng sẽ phá mốc "1/2 = phẳng" / "1 = hở", vốn là toàn bộ nội dung khoa học của chúng.
    """
    x = np.asarray(x, dtype=np.float32)
    if vrange is not None:
        lo, hi = float(vrange[0]), float(vrange[1])
    elif sym:
        a = float(np.percentile(np.abs(_interior(x, margin_px)), pct[1])) or 1.0
        lo, hi = -a, a
    else:
        lo, hi = _interior_percentile(x, pct, margin_px)
    return np.clip((x - lo) / max(hi - lo, 1e-12), 0, 1).astype(np.float32)

#### `colorize(x01, cmap)`
**Làm gì:** giá trị [0, 1] → ảnh RGB uint8 qua bảng màu matplotlib. `gray` làm trực tiếp (nhanh); đại lượng có dấu dùng
`coolwarm` (xanh = âm, đỏ = dương, trắng = 0).

In [ ]:
def colorize(x01, cmap="gray"):
    """[0,1] -> RGB uint8 qua colormap matplotlib. Dùng coolwarm cho đại lượng có dấu."""
    x01 = np.clip(np.asarray(x01, dtype=np.float32), 0, 1)
    if cmap in (None, "gray", "grey"):
        g = (x01 * 255).astype(np.uint8)
        return np.repeat(g[..., None], 3, axis=-1)
    from matplotlib import colormaps
    return (colormaps[cmap](x01)[..., :3] * 255).astype(np.uint8)

#### `_lab(img, text)`
**Làm gì:** vẽ nhãn chữ trắng trên dải đen ở đầu mỗi ô lưới. Nhãn phải là **ASCII** vì font Hershey của `cv2.putText`
không vẽ được dấu tiếng Việt.

In [ ]:
def _lab(img, text):
    """Nhãn ASCII. cv2.putText dùng font Hershey nên KHÔNG vẽ được dấu tiếng Việt."""
    img = np.ascontiguousarray(img)
    cv2.rectangle(img, (0, 0), (img.shape[1], 26), (0, 0, 0), -1)
    cv2.putText(img, text, (6, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    return img

### 10. Lưới so sánh các phương pháp
Bảng `METHODS` (mỗi phương pháp một hàm `m_*` trả ảnh hiển thị) và lưới so sánh. File `s10_grid.py`.

#### `m_depth(h, px_mm, **kw)`
**Bọc phương pháp 1** cho lưới so sánh: độ sâu cục bộ `local`, thang đối xứng, màu `coolwarm`. Mọi hàm `m_*` cùng trả dict
`{raw: số liệu, img: ảnh RGB, label: nhãn}` và nhận `**kw` để bỏ qua tham số không dùng.

In [ ]:
def m_depth(h, px_mm, **kw):
    d = depth_map(h, px_mm, hp_sigma_px=kw.get("hp_sigma_px", 12.0))
    return {"raw": d["local"], "img": colorize(norm01(d["local"], sym=True), "coolwarm"),
            "label": "1 Depth (local, detrend)"}

#### `m_normal(h, px_mm, **kw)`
**Bọc phương pháp 2:** normal map RGB với `gain` mặc định 0,35.

In [ ]:
def m_normal(h, px_mm, **kw):
    n = normals(h, px_mm, kw.get("sigma_px", 1.0), kw.get("gain", 0.35))
    return {"raw": n[..., 2], "img": normal_map_rgb(n),
            "label": "2 Normal map (gain %.2f)" % kw.get("gain", 0.35)}

#### `m_curvature(h, px_mm, **kw)`
**Bọc phương pháp 3:** hiển thị **shape index × curvedness** (điều biên), thang cố định [−1, 1].

Vì sao không hiện shape index trần: S là **tỉ số** nên bão hoà về ±1 ngay cả khi độ cong nhỏ xíu — nền khoét chỉ có
nhiễu vết đục cũng loang lổ đỏ / xanh và nuốt mất nét. Nhân với curvedness (mức độ cong) đưa vùng phẳng về trung tính mà
giữ phân loại lồi / lõm ở nơi thật sự cong (Koenderink vốn định nghĩa S đi kèm C).

In [ ]:
def m_curvature(h, px_mm, **kw):
    """
    Shape index ĐIỀU BIÊN theo curvedness.

    S là một TỈ SỐ nên nó bão hoà về +-1 ngay cả khi độ cong nhỏ vô cùng: hiện S trần thì
    nền khoét (chỉ có nhiễu vết đục) cũng đỏ/xanh loang lổ và nuốt mất nét chữ. Koenderink
    vốn định nghĩa S đi kèm C: S nói HÌNH DẠNG gì, C nói MẠNH bao nhiêu. Nhân hai cái đưa
    vùng phẳng về 0 (trung tính) và giữ nguyên phân loại lồi/lõm ở nơi thực sự có độ cong.
    """
    c = curvature(h, px_mm, kw.get("sigma_px", 1.5))
    s = c["shape_index"] * norm01(c["curvedness"], pct=(2, 98))
    return {"raw": s, "img": colorize(norm01(s, vrange=(-1, 1)), "coolwarm"),
            "label": "3 Shape index x curvedness"}

#### `m_msii(h, px_mm, **kw)`
**Bọc phương pháp 4:** MSII gộp (`combined`) với các bán kính `radii_px`, thang đối xứng, màu `coolwarm`.

In [ ]:
def m_msii(h, px_mm, **kw):
    radii = kw.get("radii_px", (3, 6, 12, 24, 48))
    c = msii_combine(msii_volume(h, px_mm, radii), radii, px_mm)["combined"]
    return {"raw": c, "img": colorize(norm01(c, sym=True), "coolwarm"), "label": "4 MSII combined"}

#### `m_ao(h, px_mm, **kw)`
**Bọc phương pháp 5:** AO hiển thị thang xám [0, 1] **không kéo giãn**. Nhận `n_az`, `radius_px`, `n_steps`.

In [ ]:
def m_ao(h, px_mm, **kw):
    ao = horizon_ao(h, px_mm, kw.get("n_az", 16), kw.get("radius_px", 24), kw.get("n_steps", 12))
    return {"raw": ao, "img": colorize(ao, "gray"), "label": "5 Ambient occlusion"}

#### `m_exaggerated(h, px_mm, **kw)`
**Bọc phương pháp 6a:** exaggerated shading với hướng đèn `ldir` (mặc định `light_dir()`), thang xám.

In [ ]:
def m_exaggerated(h, px_mm, **kw):
    s = exaggerated_shading(h, px_mm, kw.get("ldir", light_dir()), kw.get("sigma0_px", 1.0),
                            kw.get("n_scales", 4), kw.get("gain", 0.4))
    return {"raw": s, "img": colorize(s, "gray"), "label": "6a Exaggerated shading"}

#### `m_radiance(h, px_mm, **kw)`
**Bọc phương pháp 6b:** radiance scaling với `alpha` (mặc định 3), thang xám.

In [ ]:
def m_radiance(h, px_mm, **kw):
    s = radiance_scaling(h, px_mm, kw.get("ldir", light_dir()), kw.get("alpha", 3.0),
                         kw.get("sigma_px", 1.5))
    return {"raw": s, "img": colorize(s, "gray"), "label": "6b Radiance scaling"}

#### `m_lambert(h, px_mm, **kw)`
**Bọc Lambert** (đối chứng) để đưa vào lưới khi cần so sánh.

In [ ]:
def m_lambert(h, px_mm, **kw):
    s = lambert(h, px_mm, kw.get("ldir", light_dir()), kw.get("sigma_px", 1.0), kw.get("gain", 1.0))
    return {"raw": s, "img": colorize(s, "gray"), "label": "Lambert (control)"}

#### Hằng số
**Bảng tra phương pháp:** `METHODS` ánh xạ tên ngắn (`depth`, `normal`, `curvature`, `msii`, `ao`, `exaggerated`,
`radiance`, `lambert`) sang hàm `m_*` tương ứng — cell xuất hàng loạt chọn phương pháp theo tên trong `BATCH_METHODS`.
`GRID6` = 6 phương pháp mặc định của lưới so sánh.

In [ ]:
METHODS = {"depth": m_depth, "normal": m_normal, "curvature": m_curvature, "msii": m_msii,
           "ao": m_ao, "exaggerated": m_exaggerated, "radiance": m_radiance, "lambert": m_lambert}
GRID6 = ("depth", "normal", "curvature", "msii", "ao", "exaggerated")

#### `enhance_grid(h, px_mm, names, crop, cols, thumb_w, flip_mirror, **kw)`
**Làm gì:** chạy nhiều phương pháp rồi ghép thành **một ảnh lưới** RGB (mặc định 6 phương pháp, 3 cột).

**Tham số:** `names` (danh sách phương pháp), `crop = (y0, y1, x0, x1)` — **cắt trước khi tính** (nén cả mặt khắc vào
một ô lưới làm mất chi tiết nhỏ, bản crop mới là kết quả thật), `thumb_w` (bề rộng mỗi ô), `flip_mirror` (lật ngang:
mộc bản khắc ngược nên lật để chữ đọc xuôi), `**kw` chuyển tiếp cho các hàm `m_*`.

Ghép bằng numpy thay vì matplotlib: nhanh hơn và giữ nguyên độ phân giải để phóng to đọc lại.

In [ ]:
def enhance_grid(h, px_mm, names=None, crop=None, cols=3, thumb_w=520, flip_mirror=True, **kw):
    """
    Chạy nhiều phương pháp rồi ghép thành một lưới RGB uint8 (ghép bằng numpy như
    mocban_render.contact_sheet: nhanh hơn matplotlib và giữ nguyên độ phân giải để zoom).

    crop=(y0,y1,x0,x1): cắt TRƯỚC khi tính. Nén cả tấm vào một ô lưới thường làm mất chi tiết
    nhỏ - bản crop mới là kết quả thật.
    flip_mirror: mộc bản khắc NGƯỢC nên lật ngang để chữ đọc xuôi; khối không có chữ -> False.
    """
    names = list(names or GRID6)
    hh = h[crop[0]:crop[1], crop[2]:crop[3]] if crop else h
    hh = np.ascontiguousarray(hh)

    tiles = []
    for nm in names:
        img = METHODS[nm](hh, px_mm, **kw)
        rgb = img["img"]
        if flip_mirror:
            rgb = rgb[:, ::-1]
        scale = float(thumb_w) / rgb.shape[1]
        rgb = cv2.resize(rgb, (thumb_w, max(int(round(rgb.shape[0] * scale)), 1)),
                         interpolation=cv2.INTER_AREA)
        tiles.append(_lab(rgb, img["label"]))

    rows, hgt = [], max(t.shape[0] for t in tiles)
    for i in range(0, len(tiles), cols):
        row = [np.pad(t, ((0, hgt - t.shape[0]), (0, 0), (0, 0))) for t in tiles[i:i + cols]]
        while len(row) < cols:
            row.append(np.zeros_like(row[0]))
        rows.append(np.hstack(row))
    return np.vstack(rows)

## 3. Cấu hình
Mọi tham số ở một chỗ. Kết quả ghi vào `/kaggle/working/outputs/`: `soat/` (ảnh soát + `audit.csv`),
`A_render/`, `B_enhance/`, cuối cùng nén thành `outputs.zip`.

Cell này cũng thử khởi tạo pyrender và in **GPU mà OpenGL thật sự dùng**. Nếu in ra `llvmpipe` / `softpipe` thì EGL
đang render bằng CPU dù máy có GPU — phần A vẫn chạy nhưng chậm hơn nhiều.

In [ ]:
import time, json, math, shutil, csv
from pathlib import Path
import numpy as np, cv2, trimesh
import matplotlib.pyplot as plt

INPUT = Path("/kaggle/input")
OUT   = Path("/kaggle/working/outputs")
OUT_SOAT, OUT_A, OUT_B = OUT / "soat", OUT / "A_render", OUT / "B_enhance"
for d_ in (OUT_SOAT, OUT_A, OUT_B):
    d_.mkdir(parents=True, exist_ok=True)

# --- chung ---
TARGET_MM          = 200.0     # scale cạnh dài nhất của khối về N mm
RENDER_UNCONFIDENT = False     # False: bỏ qua scan chưa chắc mặt khắc (sửa manifest.csv rồi chạy lại)

# --- hướng A: render giả ảnh chụp (GPU) ---
RUN_A          = True          # False: bỏ hẳn phần A (hoặc tự bỏ khi không có GPU)
RENDER_W, RENDER_H = 1600, 1200  # độ phân giải ảnh render (giảm nếu chậm / hết RAM)
N_SHOTS        = 24            # số ảnh render MỖI scan
PRESET         = "mixed"       # mixed | topdown | handheld | raking | closeup
OCCLUDER_PROB  = 0.0           # >0 để thêm vật che 2D (tay/thước/lóa/bóng)

# --- hướng B: tăng cường hình học (CPU) ---
SCAN        = None             # scan để phân tích chi tiết (tên file mesh); None = scan đầu tiên
PX_MM       = None             # None = độ phân giải gốc của scan (đo khi nhìn thẳng mặt khắc)
FLIP_MIRROR = True             # mộc bản khắc NGƯỢC -> lật ngang cho chữ đọc xuôi; khối không có chữ: False
CROP_MM     = 50.0             # cạnh vùng zoom (mm), tự đặt vào chỗ nhiều chi tiết nhất
SIGMA0_PX   = 1.0              # chính quy hoá: xoá xung đạo hàm trên cạnh tam giác + nhiễu đo
CURV_SIGMA  = 1.5
MSII_RADII  = (3, 6, 12, 24, 48)
AO_N_AZ     = 16
AO_R_PX     = 24
LIGHT       = light_dir(35.0, 135.0)
ANGLES      = [(90, 0), (70, 25), (50, 25), (30, 25)]     # góc chiếu để xem
BATCH_ANGLES  = [(90, 0), (70, 25), (50, 25)]            # xuất hàng loạt cho mọi scan
BATCH_METHODS = ("depth", "ao", "exaggerated")
# tham số chuyển cho các hàm m_* (lưới so sánh + xuất hàng loạt) -> đổi cấu hình ở trên là mọi ảnh đổi theo
METHOD_KW = dict(ldir=LIGHT, radii_px=MSII_RADII, n_az=AO_N_AZ, radius_px=AO_R_PX, sigma0_px=SIGMA0_PX)

def show(img_or_path, title="", figsize=(15, 10), cmap=None):
    im = img_or_path if isinstance(img_or_path, np.ndarray) else \
        cv2.cvtColor(cv2.imread(str(img_or_path)), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=figsize)
    plt.imshow(im, cmap=cmap, interpolation="nearest")   # nearest: mép nét chỉ vài px, nội suy sẽ moiré
    plt.title(title); plt.axis("off"); plt.show()

def flip(a):
    return a[:, ::-1] if FLIP_MIRROR else a

def save(img_rgb, fname):
    cv2.imwrite(str(OUT_B / fname), cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))
    return OUT_B / fname

# pyrender/EGL có chạy được không (quyết định phần A và kiểu ảnh soát) + GPU nào đang render
GL = {}
try:
    _t = trimesh.creation.box(extents=[50, 50, 10]); _t.metadata["block_size_mm"] = [50, 50, 10]
    _b = PyrenderBackend(_t, 64, 64); GL = _b.gl_info(); _b.close()
    EGL_OK = True
except Exception as e:
    EGL_OK = False
    print("pyrender/EGL không chạy được (Accelerator = GPU?):", repr(e)[:200])
print(f"pyrender/EGL: {'OK' if EGL_OK else 'KHÔNG'} -> phần A {'chạy' if RUN_A and EGL_OK else 'BỎ QUA'}; phần B chạy")
if GL:
    print(f"OpenGL: {GL['renderer']} | {GL['vendor']} | {GL['version']}")
    if any(k in GL["renderer"].lower() for k in ("llvmpipe", "softpipe", "swiftshader", "software")):
        print("CẢNH BÁO: OpenGL đang render bằng CPU (không tới được driver GPU) -> phần A sẽ chậm hơn nhiều.")

## 4. Dò scan + manifest trong `/kaggle/input`
`manifest.csv` (tuỳ chọn, cột `mesh, texture, face, rot90`) ghi đè lựa chọn tự động cho từng scan — khoá là **tên file
mesh**. Có thể dùng thẳng `audit.csv` do notebook sinh ra (đã sửa) và đổi tên thành `manifest.csv`.

In [ ]:
meshes = sorted(p for p in INPUT.rglob("*") if p.suffix.lower() in MESH_EXT)
assert meshes, "Không thấy file mesh trong /kaggle/input — đã Add Input dataset chưa?"
mesh_names = [p.name for p in meshes]
assert len(set(mesh_names)) == len(mesh_names), \
    f"Trùng tên file mesh: {sorted({n for n in mesh_names if mesh_names.count(n) > 1})}"

man_files = sorted(INPUT.rglob("manifest.csv"))
manifest = read_manifest(man_files[0]) if man_files else {}
print("manifest:", man_files[0] if man_files else "không có (tự động toàn bộ)")
for p in meshes:
    print(f"  {p.relative_to(INPUT)}  ({p.stat().st_size // (1024 * 1024)} MB)  chỉ định: {manifest.get(p.name) or '-'}")

## 4b. Nạp scan + soát mặt khắc (dùng chung cho A và B)
Mỗi scan được **nạp đúng một lần** (`prepare_scan`, tốn 10–20 s với scan 1,4 triệu tam giác), soát mặt khắc
(`audit_prepared`), xoay mặt khắc lên +Z (`orient_to_face`) rồi giữ trong bộ nhớ (`SCANS`) cho phần A và phần B dùng lại.

Ảnh soát: có GPU → render nhìn thẳng từng mặt ứng viên (khung **xanh** = tự chọn & tin cậy · **vàng** = cần xem ·
**tím** = chỉ định tay); không GPU → ảnh độ sâu cục bộ của mặt được chọn (trái) và mặt đối diện (phải).
Dòng in ra cũng cho biết **texture**: `rời <tên file>`, `nhúng sẵn`, hoặc `KHÔNG CÓ` (ảnh phần A sẽ ra màu xám).

Sửa khi sai: tải `soat/audit.csv`, đổi cột `face` (vd. `C+`) / `rot90` (0–3) / `texture` cho dòng sai, lưu thành
`manifest.csv` vào dataset, chạy lại.

In [ ]:
def depth_pair_image(mesh_o):
    # ảnh soát KHÔNG cần GPU: độ sâu cục bộ nhìn thẳng mặt được chọn (trái) và mặt đối diện (phải)
    L_ = max(mesh_o.bounding_box.extents[:2])
    tiles = []
    for elev in (90, -90):
        P = project_depth(mesh_o, elev, 0, px_mm=L_ / 400)
        loc_ = depth_map(P["depth"], P["px_mm"], hp_sigma_px=6.0)["local"]
        tile = colorize(norm01(loc_, sym=True), "coolwarm"); tile[~P["valid"]] = 40
        tiles.append(cv2.resize(tile, (480, int(480 * tile.shape[0] / tile.shape[1]))))
    hh_ = max(t.shape[0] for t in tiles)
    return np.hstack([np.pad(t, ((0, hh_ - t.shape[0]), (0, 8), (0, 0)), constant_values=255) for t in tiles])

SCANS = {}      # tên file mesh -> {"path", "mesh" (mặt khắc +Z, có texture), "info", "texture"}
for p in meshes:
    t0 = time.time()
    ov = manifest.get(p.name, {})
    m_obb = prepare_scan(p, TARGET_MM, ov.get("texture", "auto"))
    t_load = time.time() - t0
    info, img = audit_prepared(m_obb, ov, image=EGL_OK)
    mesh_o = orient_to_face(m_obb, info["face"], info["rot90"])
    mesh_o.metadata["face_info"] = info
    del m_obb
    if img is None:
        img = depth_pair_image(mesh_o)
    tex = mesh_o.metadata.get("source_texture")
    SCANS[p.name] = {"path": p, "mesh": mesh_o, "info": info, "texture": tex}
    tex_msg = (f"rời {Path(tex).name}" if tex else "nhúng sẵn") if info["has_texture"] else \
        "KHÔNG CÓ -> ảnh phần A sẽ ra màu xám (thêm ảnh texture vào dataset hoặc ghi cột texture trong manifest)"
    print(f"[{'OK ' if info['confident'] else 'XEM'}] {p.name}: mặt {info['face']} — {info['reason']} "
          f"| texture: {tex_msg} | nạp {t_load:.0f}s, tổng {time.time() - t0:.0f}s")
    cv2.imwrite(str(OUT_SOAT / f"{p.stem}_faces.jpg"), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    show(img, p.name + ("" if EGL_OK else "   (trái: mặt được chọn — phải: mặt đối diện)"), figsize=(16, 6))

with open(OUT_SOAT / "audit.csv", "w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f)
    w.writerow(["mesh", "texture", "face", "rot90", "confident", "reason", "ratio", "auto_face", "has_texture", "scores"])
    for name_, s_ in SCANS.items():
        i = s_["info"]
        w.writerow([name_, Path(s_["texture"]).name if s_["texture"] else "", i["face"], i["rot90"], i["confident"],
                    i["reason"], i.get("ratio"), i["auto_face"], i["has_texture"], json.dumps(i.get("scores", {}))])
mem_mb = sum(s_["mesh"].vertices.nbytes + s_["mesh"].faces.nbytes for s_ in SCANS.values()) / 2**20
print(f"{len(SCANS)} scan, {sum(not s_['info']['confident'] for s_ in SCANS.values())} cần xem -> "
      f"{OUT_SOAT / 'audit.csv'} | mesh giữ trong RAM ≈ {mem_mb:.0f} MB (chưa tính texture)")

---
## 5. Phần A — Render giả ảnh chụp (pyrender, GPU)
`PRESET="mixed"` trộn topdown / handheld / raking / closeup: góc camera, FOV, khoảng cách, xoay trong khung, loại/hướng/
nhiệt độ màu đèn, hậu kỳ camera đều lấy ngẫu nhiên (`sample_shot`). Khung góc thấp (raking) có thể nhìn nghiêng cạnh và
tối — đúng thực tế; muốn nhiều khung thấy rõ mặt khắc thì đổi `PRESET="topdown"` hoặc `"closeup"`.
Mỗi scan có `*_meta.json`: pose camera, đèn, hậu kỳ, vật che, và `face_info` (mặt đã chọn + điểm) để truy lại.
In kèm thời gian render trung bình mỗi ảnh.

In [ ]:
if not (RUN_A and EGL_OK):
    print("BỎ QUA phần A:", "RUN_A = False" if not RUN_A else "không có GPU/EGL")
else:
    for name_, s_ in SCANS.items():
        info = s_["info"]
        if not info["confident"] and not RENDER_UNCONFIDENT:
            print(f"BỎ QUA {name_}: chưa chắc mặt khắc ({info['reason']}) — sửa manifest.csv rồi chạy lại")
            continue
        stem = s_["path"].stem
        bw, bh = s_["mesh"].metadata["block_size_mm"][:2]
        t0 = time.time()
        be = PyrenderBackend(s_["mesh"], RENDER_W, RENDER_H)
        recs = render_dataset(be, OUT_A / stem, N_SHOTS, (bw, bh), RENDER_W, RENDER_H, preset=PRESET, seed=3,
                              name=stem, occluder_prob=OCCLUDER_PROB,
                              extra_meta={"source_mesh": name_, "face_info": info})
        be.close()
        dt = time.time() - t0
        print(f"{name_}: {N_SHOTS} ảnh {RENDER_W}×{RENDER_H} trong {dt:.0f}s ({dt / N_SHOTS:.2f} s/ảnh)")
        sheet = contact_sheet([OUT_A / stem / r["file"] for r in recs[:16]], cols=4, thumb_w=420)
        cv2.imwrite(str(OUT_A / f"{stem}_sheet.jpg"), sheet); show(OUT_A / f"{stem}_sheet.jpg", figsize=(18, 14))

### (Tuỳ chọn) Thử vật che 2D
Giả tay/thước/nhãn băng dính/lóa flash/bóng người chụp trên scan đầu tiên. Đây là **ảnh demo**, không phải dữ liệu chính;
muốn áp cho cả bộ thì đặt `OCCLUDER_PROB` > 0 ở cell cấu hình.

In [ ]:
demo = next((s_ for s_ in SCANS.values() if s_["info"]["confident"] or RENDER_UNCONFIDENT), None)
if RUN_A and EGL_OK and demo is not None:
    bw, bh = demo["mesh"].metadata["block_size_mm"][:2]
    be = PyrenderBackend(demo["mesh"], RENDER_W, RENDER_H)
    rc = render_dataset(be, OUT_A / "occluder_demo", 8, (bw, bh), RENDER_W, RENDER_H, preset=PRESET, seed=9,
                        name="occ", occluder_prob=0.8)
    be.close()
    sheet = contact_sheet([OUT_A / "occluder_demo" / r["file"] for r in rc], cols=4, thumb_w=420)
    cv2.imwrite(str(OUT_A / "occluder_demo_sheet.jpg"), sheet); show(OUT_A / "occluder_demo_sheet.jpg", figsize=(18, 8))
    print("vật che mỗi ảnh:", [r["occluders"] for r in rc])
else:
    print("bỏ qua (phần A không chạy)")

---
## 6. Phần B — Tăng cường hình học, độc lập ánh sáng (CPU)

```
mặt khắc hướng +Z ──chiếu trực giao (Z-buffer, rasterize tam giác)──> ảnh độ sâu 2D ──tăng cường──> ảnh kết quả
                     elev/azim/roll tuỳ ý                                (mm)            6 phương pháp
```

Ảnh độ sâu là mặt Monge `z = f(u,v)` trong hệ quy chiếu camera, nên toàn bộ hình học vi phân chạy bằng
numpy/cv2/scipy thay cho tính toán trên mesh hàng triệu mặt.

| # | Phương pháp | Công thức lõi |
|---|---|---|
| 1 | Bản đồ độ sâu | `h - G_σ * h` (khử nền thấp tần) |
| 2 | Bản đồ pháp tuyến | `n = normalize(-hx, -hy, 1)` |
| 3 | Độ cong (Monge) | `H = ((1+q²)r - 2pqs + (1+p²)t) / (2W³)` |
| 4 | MSII | `v_r ≈ ½ + 3/(4r)·(trung_bình_đĩa_r(h) - h)` |
| 5 | Ambient Occlusion | `AO = trung_bình_φ[ 1/(1 + tan²θ_h) ]` |
| 6 | Exaggerated Shading + Radiance Scaling | `S_i = ½ + c·(s_i - s_{i+1})`; `σ(κ̄) = ((α+1)+(α-1)κ̄)/((α+1)-(α-1)κ̄)` |

Phân tích chi tiết chạy trên **một** scan (`SCAN`); cuối phần B xuất ảnh tăng cường cho **mọi** scan. Mesh lấy từ bộ
nhớ (`SCANS`), không nạp lại.

### Chọn scan phân tích
Kiểm bằng mắt: **trái** là mặt được chọn (phải có nét khắc), **phải** là mặt đối diện.

In [ ]:
SCAN_NAME = SCAN or next(iter(SCANS))
mesh, fi = SCANS[SCAN_NAME]["mesh"], SCANS[SCAN_NAME]["info"]
print(f"{SCAN_NAME}: {len(mesh.vertices):,} đỉnh, {len(mesh.faces):,} mặt, khối(mm) {np.round(mesh.bounding_box.extents, 1)}")
print(f"mặt khắc {fi['face']} — {fi['reason']}" + ("" if fi["confident"] else "   <<< CHƯA CHẮC: kiểm ảnh bên dưới"))

P_top = project_depth(mesh, 90, 0, px_mm=PX_MM)
PX_MM = P_top["px_mm"]                                # mọi phép chiếu sau dùng CHUNG bước lưới này
P_bot = project_depth(mesh, -90, 0, px_mm=PX_MM)
print(f"px_mm = {PX_MM:.4f} mm/px -> ảnh nhìn thẳng {P_top['depth'].shape}")

fig, ax = plt.subplots(1, 2, figsize=(16, 7))
for a_, P, t_ in zip(ax, (P_top, P_bot), ("mặt được chọn (+Z)", "mặt đối diện (-Z)")):
    loc = depth_map(P["depth"], PX_MM, hp_sigma_px=12.0)["local"]
    s_ = float(np.percentile(np.abs(loc[P["valid"]]), 99)) or 1.0
    a_.imshow(np.where(P["valid"], loc, np.nan), cmap="coolwarm", vmin=-s_, vmax=s_, interpolation="nearest")
    a_.set_title(f"{t_}: độ sâu cục bộ ±{s_:.2f} mm"); a_.axis("off")
plt.tight_layout(); plt.show()

### Kiểm chứng giải tích
Trước khi tin bất kỳ bức ảnh nào, kiểm các đại lượng có **đáp án đóng**: phép chiếu (mặt sin biết trước độ cao, mặt phẳng nhìn xiên biết trước độ dốc) và các phép tăng cường (bán cầu, mặt phẳng, chân tường, khai triển Pottmann, hàm Möbius). Bắt trọn các lỗi đường ống hay gặp nhất (đảo trục, quên chia `px_mm`, nhân bậc 2 không tổng bằng 0) chỉ trong một lần chạy. Có bài FAIL thì cell dừng (`assert`).

In [ ]:
ok = True
def chk(label, cond, detail=""):
    global ok; ok = ok and bool(cond)
    print(f"  [{'PASS' if cond else 'FAIL'}] {label}  {detail}")

def grid_mesh(zfun, n, step):
    xs = (np.arange(n) - (n - 1) / 2) * step
    X, Y = np.meshgrid(xs, xs)
    Vg = np.stack([X, Y, zfun(X, Y)], -1).reshape(-1, 3)
    idx = np.arange(n * n).reshape(n, n)
    a, b, c, d = idx[:-1, :-1].ravel(), idx[:-1, 1:].ravel(), idx[1:, :-1].ravel(), idx[1:, 1:].ravel()
    m = trimesh.Trimesh(Vg, np.concatenate([np.stack([a, b, c], 1), np.stack([b, d, c], 1)]), process=False)
    if (m.face_normals[:, 2] < 0).mean() > 0.5:
        m.invert()
    return m

# 0a) phép chiếu, mặt sin: rasterize lấy độ sâu đúng tại tâm pixel, sai số chỉ còn là sai số NỘI SUY TUYẾN TÍNH trên
#     tam giác lưới bước h, cỡ 2·(|f_xx| + |f_yy|)max·h²/8 (hệ số 2 cho cạnh chéo của tam giác)
zf = lambda x, y: np.sin(2 * np.pi * x / 20) * np.cos(2 * np.pi * y / 15)
gm = grid_mesh(zf, 241, 0.5)
P = project_depth(gm, 90, 0, px_mm=0.5); dd, px = P["depth"], P["px_mm"]
Vg = np.asarray(gm.vertices)
Xc, Yc = np.meshgrid(Vg[:, 0].min() + (np.arange(dd.shape[1]) + 0.5) * px, Vg[:, 1].max() - (np.arange(dd.shape[0]) + 0.5) * px)
zc = zf(Xc, Yc); sl = (slice(5, -5), slice(5, -5))
err = np.abs((dd - dd[sl].mean()) - (zc - zc[sl].mean()))[sl]
bound = 2 * ((2 * np.pi / 20) ** 2 + (2 * np.pi / 15) ** 2) * 0.5 ** 2 / 8
chk("chiếu mặt sin: trung vị sai số < 0.01 mm", np.median(err) < 0.01, f"{np.median(err):.4f} mm")
chk("chiếu mặt sin: sai số lớn nhất < giới hạn nội suy", err.max() < bound, f"{err.max():.4f} < {bound:.4f} mm")

# 0b) phép chiếu, mặt phẳng z=0 nhìn xiên: độ sâu tăng theo cột đúng px/tan(elev), không đổi theo hàng
pl = grid_mesh(lambda x, y: 0 * x, 101, 1.0)
for elev in (60.0, 30.0):
    P = project_depth(pl, elev, 0, px_mm=0.5)
    r_, c_ = np.nonzero(P["valid"])
    kk = np.linalg.lstsq(np.stack([c_, r_, np.ones_like(c_)], 1).astype(float), P["depth"][r_, c_], rcond=None)[0]
    pred = 0.5 / math.tan(math.radians(elev))
    chk(f"chiếu mặt phẳng elev={elev:.0f}°: dốc theo cột = px/tan(elev)", abs(kk[0] - pred) / pred < 0.005,
        f"{kk[0]:.5f} vs {pred:.5f}")
    chk(f"chiếu mặt phẳng elev={elev:.0f}°: không dốc theo hàng", abs(kk[1]) < 1e-3, f"{kk[1]:+.6f}")

# 1) Bán cầu bán kính R:  H = -1/R,  K = +1/R^2  (quy ước Monge: phần NỔI có H < 0)
for px_mm, R_px in [(0.386, 120), (1.0, 120)]:
    N = 2*R_px + 41; cc = N // 2
    yy, xx = np.mgrid[0:N, 0:N].astype(np.float64)
    R_mm = R_px * px_mm
    hs = np.sqrt(np.maximum(R_mm**2 - ((xx-cc)**2 + (yy-cc)**2) * px_mm**2, 0.0)).astype(np.float32)
    cv_ = curvature(hs, px_mm, sigma_px=1.5)
    Hc, Kc = cv_["H"][cc, cc], cv_["K"][cc, cc]
    chk(f"bán cầu px_mm={px_mm}  H=-1/R", abs(Hc + 1/R_mm)/(1/R_mm) < 0.05, f"{Hc:+.5f} vs {-1/R_mm:+.5f}")
    chk(f"bán cầu px_mm={px_mm}  K=+1/R²", abs(Kc - 1/R_mm**2)/(1/R_mm**2) < 0.05, f"{Kc:+.6f} vs {1/R_mm**2:+.6f}")

# 2) Mặt phẳng: AO = 1, MSII v_r = 1/2, H = K = 0
flat = np.zeros((200, 200), np.float32)
chk("mặt phẳng AO = 1", abs(horizon_ao(flat, 0.386, 16, 24)[100,100] - 1) < 1e-5)
chk("mặt phẳng MSII = ½", np.abs(msii_volume(flat, 0.386, (6,12,24))[100,100] - 0.5).max() < 1e-5)
chk("mặt phẳng H = 0", abs(curvature(flat, 0.386)["H"][100,100]) < 1e-6)

# 3) Chân tường đứng: 7/16 phương vị bị che -> AO ≈ 0.5625
wall = np.zeros((200, 200), np.float32); wall[:, 100:] = 50.0
ao_w = horizon_ao(wall, 1.0, 16, 24)
chk("chân tường AO ≈ 0.5", 0.45 < ao_w[100, 99] < 0.62, f"{ao_w[100,99]:.4f}")
chk("xa tường  AO = 1",    abs(ao_w[100, 10] - 1) < 1e-4, f"{ao_w[100,10]:.6f}")

# 4) MSII khớp khai triển Pottmann v_r - ½ = (3/16)·H·r trên vòm cầu
NN, c0, px_, R_ = 401, 200, 0.386, 60.0
y2, x2 = np.mgrid[0:NN, 0:NN].astype(np.float64)
dome = np.sqrt(np.maximum(R_**2 - ((x2-c0)**2 + (y2-c0)**2) * px_**2, 0.0)).astype(np.float32)
Vd = msii_volume(dome, px_, (12, 24))
for j, rpx in enumerate((12, 24)):
    pred = (3/16) * (-1/R_) * (rpx * px_); got = Vd[c0, c0, j] - 0.5
    chk(f"Pottmann r={rpx}px", abs(got-pred)/abs(pred) < 0.10, f"{got:+.5f} vs {pred:+.5f}")

# 5) Radiance scaling: ba tính chất xác định hàm Möbius
for al in (2.0, 3.0, 5.0):
    sig_ = lambda kb: ((al+1) + (al-1)*kb) / ((al+1) - (al-1)*kb)
    chk(f"σ(κ̄) với α={al}", abs(sig_(0)-1) < 1e-12 and abs(sig_(1)-al) < 1e-12 and abs(sig_(-1)-1/al) < 1e-12)

print("\n" + ("TẤT CẢ PASS" if ok else "CÓ TEST FAIL — dừng lại và sửa trước khi đọc ảnh"))
assert ok

### Bước 1 — Chiếu scan từ nhiều góc

Khi `elev` giảm, ảnh **co ngắn** theo phương nghiêng (foreshortening) và tỉ lệ pixel bị che khuất tăng dần — đó
là hình học đúng, không phải lỗi. Mọi góc dùng **chung** bước lưới `PX_MM` để so sánh được.

Chú ý: nét khắc vẫn **đọc được ở mọi góc**, vì tăng cường dựa trên hình học chứ không dựa vào đèn.

In [ ]:
proj = {}
for elev, azim in ANGLES:
    t0 = time.time(); P = project_depth(mesh, elev_deg=elev, azim_deg=azim, px_mm=PX_MM)
    proj[(elev, azim)] = P
    print(f"elev={elev:3d}° azim={azim:3d}°  ảnh {str(P['depth'].shape):14s} "
          f"hợp lệ {P['valid'].mean()*100:5.1f}%  dải độ sâu {P['depth'].max():7.2f} mm  "
          f"{(time.time()-t0)*1000:5.0f} ms")

fig, ax = plt.subplots(1, len(ANGLES), figsize=(4.3*len(ANGLES), 5.5))
for a_, (key_, P) in zip(ax, proj.items()):
    dep = np.where(P["valid"], P["depth"], np.nan)        # để trống chỗ bị che, không bôi nhoè
    a_.imshow(flip(dep), cmap="viridis", interpolation="nearest")
    a_.set_title(f"elev={key_[0]}° azim={key_[1]}°\nảnh độ sâu đã chiếu"); a_.axis("off")
plt.tight_layout(); plt.show()

# tăng cường (AO) trên chính các ảnh đã chiếu
fig, ax = plt.subplots(1, len(ANGLES), figsize=(4.3*len(ANGLES), 5.5))
for a_, (key_, P) in zip(ax, proj.items()):
    hh = prepare_height(P["depth"], P["px_mm"], SIGMA0_PX)
    ao_ = np.where(P["valid"], horizon_ao(hh, P["px_mm"], 12, AO_R_PX), np.nan)
    a_.imshow(flip(ao_), cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    a_.set_title(f"elev={key_[0]}° — AO sau khi chiếu"); a_.axis("off")
plt.tight_layout(); plt.show()

### Làm trơn σ₀, đo độ sâu nét khắc, tự chọn vùng zoom

Ảnh độ sâu của scan là mặt ghép từ các **tam giác phẳng** cộng nhiễu đo; đạo hàm bậc hai của nó là dãy xung trên cạnh
tam giác, nên phải làm trơn σ₀ ≈ 1 px trước (xem `prepare_height`).

Vùng zoom (`CROP_MM`) được **tự đặt vào chỗ nhiều nét khắc nhất**: mật độ vách dốc ngắn (độ dốc > 1), sau khi bỏ các
đường thẳng dài (khe nứt, mép khối) bằng phép mở hình thái và bỏ dải mép 5 %; cửa sổ phải nằm trọn trong lõi mặt khắc.
Chọn theo biên độ độ sâu thì cửa sổ sẽ rơi vào khe nứt (chỗ sâu nhất).

In [ ]:
# Ảnh độ sâu dùng cho toàn bộ phần tăng cường = KẾT QUẢ CHIẾU. Đổi VIEW là mọi phương pháp chạy lại ở góc đó.
VIEW = (90, 0)
P_view = proj[VIEW] if VIEW in proj else project_depth(mesh, *VIEW, px_mm=PX_MM)
h_raw, VALID = P_view["depth"], P_view["valid"]
h = prepare_height(h_raw, PX_MM, sigma0_px=SIGMA0_PX)
loc = depth_map(h, PX_MM, hp_sigma_px=12.0)["local"]
# biên độ nét khắc ĐIỂN HÌNH: p5–p95, không p1–p99 — vài khe sâu (vd. khe nứt xuyên khối, rãnh ghép) chiếm
# chưa tới 1% diện tích nhưng sâu tới đáy khối và sẽ thổi phồng con số này
RELIEF_MM = float(np.subtract(*np.percentile(loc[VALID], [95, 5])))
st = relief_stats(h_raw, PX_MM)
print(f"góc nhìn {VIEW}  ->  ảnh độ sâu {h_raw.shape}")
for key_, v in st.items():
    print(f"  {key_:12s} {v}")
print(f"  relief nét khắc điển hình (p5–p95 sau khử nền) = {RELIEF_MM:.3f} mm ≈ {RELIEF_MM / PX_MM:.1f} px")
print(f"  (p1–p99 = {np.subtract(*np.percentile(loc[VALID], [99, 1])):.3f} mm: lớn hơn nhiều nếu có khe sâu)")

# vùng zoom: cửa sổ CROP_MM quanh chỗ chi tiết DÀY nhất, nằm trọn trong mặt khắc. Chi tiết = mật độ VÁCH DỐC NGẮN
# (> 45°): đo biên độ độ sâu thì một khe sâu thắng; còn cấu trúc thẳng DÀI (khe ghép, mép khối, đường kẻ cột) bị tách
# ra bằng phép mở với phần tử đoạn thẳng ngang / dọc dài CROP/3 — nét chữ, hoa văn là các vách ngắn nên giữ lại.
cpx = max(int(CROP_MM / PX_MM), 32)
dv0 = derivatives(h, PX_MM, 1.0)
steep = ((np.hypot(dv0["hx"], dv0["hy"]) > 1.0) & VALID).astype(np.uint8)
Lz = max(cpx // 3, 9)
lines = cv2.morphologyEx(steep, cv2.MORPH_OPEN, np.ones((1, Lz), np.uint8)) | \
        cv2.morphologyEx(steep, cv2.MORPH_OPEN, np.ones((Lz, 1), np.uint8))
lines = cv2.dilate(lines, np.ones((5, 5), np.uint8))
rim = max(int(0.05 * max(h.shape)), 3) | 1           # bỏ viền khối (gờ mép cũng là vách ngắn nhưng không phải nội dung)
core = cv2.erode(np.pad(VALID, 1).astype(np.uint8), np.ones((rim, rim), np.uint8))[1:-1, 1:-1]
# tâm hợp lệ = cả cửa sổ nằm trong lõi. Đệm False quanh ảnh: cv2.erode mặc định coi ngoài ảnh là hợp lệ.
kw_ = cpx // 2 * 2 + 1
inner = cv2.erode(np.pad(core, kw_), np.ones((kw_, kw_), np.uint8))[kw_:-kw_, kw_:-kw_].astype(bool)
en = cv2.GaussianBlur((steep & (1 - lines) & core).astype(np.float32), (0, 0), cpx / 4)
en[~inner] = 0
cy, cx = np.unravel_index(int(np.argmax(en)), en.shape) if inner.any() else (h.shape[0] // 2, h.shape[1] // 2)
CROP = tuple(int(v) for v in (max(cy - cpx // 2, 0), min(cy + cpx // 2, h.shape[0]),
                              max(cx - cpx // 2, 0), min(cx + cpx // 2, h.shape[1])))
cr = lambda a: flip(a[CROP[0]:CROP[1], CROP[2]:CROP[3]])
print("vùng zoom (y0, y1, x0, x1):", CROP, f"= {cpx * PX_MM:.0f} mm")

# mặt cắt ngang qua tâm vùng zoom: thấy ngay relief, độ rộng mép, và nhiễu trong một hình
seg = slice(CROP[2], CROP[3])
plt.figure(figsize=(14, 3.2))
plt.plot(np.arange(seg.start, seg.stop), h_raw[cy, seg] - np.median(h_raw[cy, seg]), lw=1.0, label="thô")
plt.plot(np.arange(seg.start, seg.stop), h[cy, seg] - np.median(h[cy, seg]), lw=1.8, label=f"sau σ₀={SIGMA0_PX}px")
plt.axhline(0, color="k", lw=0.6)
plt.xlabel("cột (px)"); plt.ylabel("độ cao (mm)")
plt.title(f"Mặt cắt ngang hàng {cy} — px_mm={PX_MM:.4f}")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()
show(cr(np.where(VALID, loc, np.nan)), "vùng zoom — độ sâu cục bộ", figsize=(7, 7), cmap="coolwarm")

### Phương pháp 1 — Bản đồ độ sâu

Bản thô có đơn vị vật lý (mm) nhưng bị chi phối bởi dạng tổng thể của khối (cong, vênh, nghiêng) — nét khắc chỉ là
phần nhỏ trong dải màu. Bản `local` (trừ nền thấp tần) mới là bản dùng được, và nó cũng là bản chịu được khối cong/vênh.

In [ ]:
dm = depth_map(h, PX_MM, hp_sigma_px=12.0)
print(f"độ cao: [{dm['lo_mm']:.3f}, {dm['hi_mm']:.3f}] mm (percentile 1–99, đã loại viền)")

fig, ax = plt.subplots(1, 2, figsize=(16, 9))
im = ax[0].imshow(cr(dm["mm"]), cmap="viridis", interpolation="nearest")
ax[0].set_title("độ cao thô [mm]"); ax[0].axis("off"); fig.colorbar(im, ax=ax[0], fraction=.046, label="mm")
amp = float(np.percentile(np.abs(dm["local"][VALID]), 99))
ax[1].imshow(cr(dm["local"]), cmap="coolwarm", vmin=-amp, vmax=amp, interpolation="nearest")
ax[1].set_title(f"local = h - G₁₂ₚₓ*h   ±{amp:.3f} mm"); ax[1].axis("off")
plt.tight_layout(); plt.show()

save(flip(colorize(norm01(dm["local"], sym=True), "coolwarm")), "01_depth_local.png")

### Phương pháp 2 — Bản đồ pháp tuyến

Mép nét khắc dốc hơn nhiều so với nền nên normal map **thô** thường cho ra nền một màu cộng viền cháy sáng — đúng về
kỹ thuật nhưng khó đọc. Hệ số `gain < 1` nén độ dốc lại. Kèm panel **slope/aspect HSV**: mắt người phân tách nét
theo *hướng dốc*, nên bản này thường đọc rõ hơn normal map RGB.

In [ ]:
for g in (1.0, 0.35):
    n = normals(h, PX_MM, sigma_px=1.0, gain=g)
    show(cr(normal_map_rgb(n)), f"normal map, gain={g}" + ("  (thô)" if g == 1.0 else "  (đã nén độ dốc)"),
         figsize=(8, 8))

hsv = slope_aspect_hsv(h, PX_MM, sigma_px=1.0)
show(cr(hsv), "slope / aspect (HSV): hue = hướng dốc", figsize=(8, 8))
save(flip(normal_map_rgb(normals(h, PX_MM, 1.0, 0.35))), "02_normal.png")
save(flip(hsv), "02_slope_aspect.png")

### Phương pháp 3 — Độ cong (Monge patch chính xác)

`H ≈ ½∇²h` **chỉ** đúng khi độ dốc ≲ 0.3. Cell dưới đo độ dốc thật của scan và tỉ số lệch của xấp xỉ Laplacian
trên 1% pixel cong nhất. Tỉ số có thể gần 1 ngay cả khi độ dốc rất lớn: pixel cong nhất thường nằm ở đỉnh/chân vách,
nơi độ dốc cục bộ lại nhỏ — công thức chính xác đúng ở mọi nơi nên không phải đoán trường hợp nào.

`K` được hiện cho đủ nhưng **không** vào lưới so sánh: nét khắc gần *developable* (sống + vách) nên `K ≈ 0` gần khắp
nơi và panel K chủ yếu là nhiễu.

In [ ]:
cvt = curvature(h, PX_MM, sigma_px=CURV_SIGMA)
dv = derivatives(h, PX_MM, CURV_SIGMA)
lap_H = 0.5 * (dv["hxx"] + dv["hyy"])                      # xấp xỉ Laplacian
edge = VALID & (np.abs(cvt["curvedness"]) > np.percentile(np.abs(cvt["curvedness"][VALID]), 99))
print(f"trên 1% pixel cong nhất: |Laplacian/2| / |H| trung vị = "
      f"{np.median(np.abs(lap_H[edge]) / (np.abs(cvt['H'][edge]) + 1e-9)):.1f}×  <- sai số nếu dùng Laplacian")
print(f"độ dốc p99 = {np.percentile(np.hypot(dv['hx'], dv['hy'])[VALID], 99):.2f}  (xấp xỉ Laplacian cần ≲ 0.3)")

fig, ax = plt.subplots(1, 3, figsize=(17, 6))
aH = float(np.percentile(np.abs(cvt["H"][VALID]), 99))
ax[0].imshow(cr(cvt["H"]), cmap="coolwarm", vmin=-aH, vmax=aH, interpolation="nearest")
ax[0].set_title(f"H [1/mm]  ±{aH:.2f}\n(âm = phần nổi)")
aK = float(np.percentile(np.abs(cvt["K"][VALID]), 99))
ax[1].imshow(cr(cvt["K"]), cmap="coolwarm", vmin=-aK, vmax=aK, interpolation="nearest")
ax[1].set_title(f"K [1/mm²]  ±{aK:.3f}")
ax[2].imshow(cr(cvt["shape_index"] * norm01(cvt["curvedness"])), cmap="coolwarm", vmin=-1, vmax=1,
             interpolation="nearest")
ax[2].set_title("shape index × curvedness\n(điều biên để nền phẳng về trung tính)")
for a_ in ax: a_.axis("off")
plt.tight_layout(); plt.show()

save(flip(METHODS["curvature"](h, PX_MM, **METHOD_KW)["img"]), "03_curvature.png")

### Phương pháp 4 — Bất biến tích phân đa tỉ lệ (MSII)

`v_r(p)` = tỉ lệ thể tích khối nằm trong quả cầu bán kính `r` quanh điểm. Mặt phẳng cho đúng `½`; khai triển
Pottmann cho `v_r − ½ = (3/16)·H·r` (quy ước Monge của module này).

Xấp xỉ convolution chỉ hợp lệ khi `r_mm ≳ 2·relief`; bán kính nhỏ hơn thì **bão hoà** và chỉ nên đọc định tính.
Cell dưới dùng biên độ nét khắc đo được trên chính scan (`RELIEF_MM`) và đối chiếu xấp xỉ với cầu phương chính xác.

In [ ]:
Vm = msii_volume(h, PX_MM, MSII_RADII)
comb = msii_combine(Vm, MSII_RADII, PX_MM)

print(f"relief nét khắc đo được = {RELIEF_MM:.3f} mm")
print("r(px)   r(mm)   max|v-½| lý thuyết   đo được")
for j, r in enumerate(MSII_RADII):
    r_mm = r * PX_MM
    print(f"{r:5d} {r_mm:7.2f} {3*RELIEF_MM/(4*r_mm):16.2f} {np.abs(Vm[...,j]-0.5)[VALID].max():11.3f}"
          + ("   <- bão hoà" if 3*RELIEF_MM/(4*r_mm) > 0.4 else ""))

# thang màu KHOÁ CHUNG, nếu không mỗi panel tự chuẩn hoá và dãy sweep chẳng cho thấy gì
amp = float(np.percentile(np.abs(Vm[..., len(MSII_RADII)//2] - 0.5)[VALID], 99))
fig, ax = plt.subplots(1, len(MSII_RADII), figsize=(4*len(MSII_RADII), 4.6))
for j, r in enumerate(MSII_RADII):
    ax[j].imshow(cr(Vm[..., j] - 0.5), cmap="coolwarm", vmin=-amp, vmax=amp, interpolation="nearest")
    ax[j].set_title(f"r = {r} px"); ax[j].axis("off")
plt.suptitle(f"v_r − ½  (thang chung ±{amp:.3f})"); plt.tight_layout(); plt.show()

show(cr(comb["scale_argmax"]), "tỉ lệ đặc trưng (argmax |v−½|)", figsize=(7, 7), cmap="turbo")

if 24 in MSII_RADII:
    Ve = msii_volume(h[CROP[0]:CROP[1], CROP[2]:CROP[3]], PX_MM, (24,), exact=True, n_rho=8, n_phi=16)
    Vc = Vm[CROP[0]:CROP[1], CROP[2]:CROP[3], MSII_RADII.index(24)]
    print(f"\nr=24px  xấp xỉ convolution vs cầu phương chính xác: "
          f"lệch trung vị {np.median(np.abs(Ve[...,0]-Vc)):.5f}, max {np.abs(Ve[...,0]-Vc).max():.5f}")
save(flip(METHODS["msii"](h, PX_MM, **METHOD_KW)["img"]), "04_msii.png")

### Phương pháp 5 — Ambient Occlusion (horizon mapping)

`AO = trung_bình_φ[ 1/(1+tan²θ_h) ]` (xem `horizon_ao`). Rãnh khắc và vết nứt thường hiện rõ nhất ở đây. Bán kính tìm
chân trời quyết định thang chi tiết được làm nổi — cell dưới quét vài bán kính trên **cùng thang [0,1]** để chọn
`AO_R_PX` cho scan mới.

In [ ]:
t0 = time.time(); ao = horizon_ao(h, PX_MM, AO_N_AZ, AO_R_PX); print(f"AO: {time.time()-t0:.2f}s")
print(f"AO ∈ [{ao[VALID].min():.3f}, {ao[VALID].max():.3f}]  (1 = hở hoàn toàn)")

fig, ax = plt.subplots(1, 2, figsize=(16, 9))
ax[0].imshow(flip(np.where(VALID, ao, np.nan)), cmap="gray", vmin=0, vmax=1, interpolation="nearest")  # KHÔNG kéo giãn
ax[0].set_title("AO toàn mặt khắc  [0,1] không kéo giãn"); ax[0].axis("off")
ax[1].imshow(cr(ao), cmap="gray", vmin=0, vmax=1, interpolation="nearest")
ax[1].set_title("AO — zoom"); ax[1].axis("off")
plt.tight_layout(); plt.show()

ao_radii = (8, AO_R_PX, 3 * AO_R_PX)
fig, ax = plt.subplots(1, len(ao_radii), figsize=(5.5 * len(ao_radii), 6))
for a_, r in zip(ax, ao_radii):
    a_.imshow(cr(horizon_ao(h, PX_MM, 12, r)), cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    a_.set_title(f"AO bán kính {r} px = {r * PX_MM:.1f} mm"); a_.axis("off")
plt.tight_layout(); plt.show()
save(flip(colorize(ao, "gray")), "05_ao.png")

### Phương pháp 6 — Exaggerated Shading & Radiance Scaling

**6a — Exaggerated Shading** tô bóng phần **chi tiết** mà mỗi tỉ lệ thêm vào so với tỉ lệ thô hơn, nên gần như không
đổi theo góc đèn. **6b — Radiance Scaling** nhân bóng Lambert với hàm Möbius của độ cong (lồi sáng lên, lõm tối đi).
Chi tiết công thức xem giải thích của `exaggerated_shading` và `radiance_scaling` ở mục 2B.

**Lambert thuần là đối chứng** — hình thứ hai so Lambert và exaggerated ở ba góc đèn 10°, 25°, 60°.

In [ ]:
lam = lambert(h, PX_MM, LIGHT)
exg = exaggerated_shading(h, PX_MM, LIGHT, SIGMA0_PX, n_scales=4, gain=0.4)
rad = radiance_scaling(h, PX_MM, LIGHT, alpha=3.0, sigma_px=CURV_SIGMA)
for nm, im_ in [("Lambert (đối chứng)", lam), ("6a Exaggerated", exg), ("6b Radiance scaling", rad)]:
    print(f"{nm:24s} dải [{im_.min():.3f}, {im_.max():.3f}]")

fig, ax = plt.subplots(1, 3, figsize=(17, 6))
for a_, im_, t_ in zip(ax, (lam, exg, rad), ("Lambert (đối chứng)", "6a Exaggerated", "6b Radiance scaling")):
    a_.imshow(cr(im_), cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    a_.set_title(t_); a_.axis("off")
plt.tight_layout(); plt.show()

# điểm mấu chốt: ở góc chiếu THẤP Lambert sập, exaggerated shading thì không
fig, ax = plt.subplots(2, 3, figsize=(17, 11))
for j, elev in enumerate((10.0, 25.0, 60.0)):
    Ld = light_dir(elev, 135.0)
    ax[0, j].imshow(cr(lambert(h, PX_MM, Ld)), cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    ax[0, j].set_title(f"Lambert — elev {elev:.0f}°")
    ax[1, j].imshow(cr(exaggerated_shading(h, PX_MM, Ld, SIGMA0_PX)), cmap="gray", vmin=0, vmax=1,
                    interpolation="nearest")
    ax[1, j].set_title(f"Exaggerated — elev {elev:.0f}°")
for a_ in ax.ravel(): a_.axis("off")
plt.tight_layout(); plt.show()

save(flip(colorize(exg, "gray")), "06a_exaggerated.png")
save(flip(colorize(rad, "gray")), "06b_radiance.png")

### Lưới so sánh sáu phương pháp
Lưới toàn mặt khắc là **bối cảnh**: nén cả mặt vào một ô lưới thường mất chi tiết nhỏ. Lưới **zoom** ngay dưới mới là kết quả thật. Cả hai đều được ghi ra PNG nguyên độ phân giải để phóng to đọc lại.

In [ ]:
t0 = time.time()
g_full = enhance_grid(h, PX_MM, thumb_w=460, flip_mirror=FLIP_MIRROR, **METHOD_KW)
print(f"lưới toàn mặt {g_full.shape}  ({time.time()-t0:.1f}s)")
save(g_full, "grid_full.png"); show(g_full, "Sáu phương pháp — toàn mặt khắc", figsize=(17, 15))

In [ ]:
g_crop = enhance_grid(h, PX_MM, crop=CROP, thumb_w=420, flip_mirror=FLIP_MIRROR, **METHOD_KW)
save(g_crop, "grid_crop.png")
show(g_crop, f"Sáu phương pháp — vùng zoom {CROP_MM:.0f} mm  ← hình quan trọng nhất", figsize=(17, 12))

### Xuất ảnh tăng cường cho mọi scan
Chạy `BATCH_METHODS` × `BATCH_ANGLES` cho **mọi scan** (mesh lấy từ bộ nhớ, mặt khắc từ bước soát chung).
Pixel ngoài khối / bị che khuất (`valid = False`) được tô **đen** để không đưa dữ liệu lấp giả vào tập huấn luyện.
Mỗi scan có `batch_meta.json` ghi mặt khắc, góc chiếu, `px_mm`.

In [ ]:
BATCH = OUT_B / "batch"; BATCH.mkdir(exist_ok=True)
for name_, s_ in SCANS.items():
    fi_ = s_["info"]
    if not fi_["confident"] and not RENDER_UNCONFIDENT:
        print(f"BỎ QUA {name_}: chưa chắc mặt khắc ({fi_['reason']}) — sửa manifest.csv rồi chạy lại")
        continue
    t0 = time.time()
    m = s_["mesh"]
    px = PX_MM if name_ == SCAN_NAME else project_depth(m, 90, 0)["px_mm"]   # độ phân giải gốc từng scan
    stem = s_["path"].stem
    d_ = BATCH / stem; d_.mkdir(exist_ok=True)
    meta = {"mesh": name_, "face_info": fi_, "px_mm": px, "flip_mirror": FLIP_MIRROR, "images": []}
    for elev, azim in BATCH_ANGLES:
        P = project_depth(m, elev, azim, px_mm=px)
        hh = prepare_height(P["depth"], px, SIGMA0_PX)
        for nm in BATCH_METHODS:
            img = METHODS[nm](hh, px, **METHOD_KW)["img"].copy()
            img[~P["valid"]] = 0
            fn = f"{stem}_e{elev}_a{azim}_{nm}.png"
            cv2.imwrite(str(d_ / fn), cv2.cvtColor(flip(img), cv2.COLOR_RGB2BGR))
            meta["images"].append({"file": fn, "elev": elev, "azim": azim, "method": nm,
                                   "valid_frac": float(P["valid"].mean())})
    (d_ / "batch_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"{name_}: {len(meta['images'])} ảnh ({time.time() - t0:.0f}s)")

---
## 7. Đóng gói kết quả
`outputs.zip` gồm `soat/`, `A_render/` (nếu phần A chạy) và `B_enhance/`.

In [ ]:
shutil.make_archive("/kaggle/working/outputs", "zip", OUT)
for sub in (OUT_SOAT, OUT_A, OUT_B):
    files = [q for q in sub.rglob("*") if q.is_file()]
    print(f"{sub.name:10s} {len(files):5d} file  {sum(q.stat().st_size for q in files) // 1024:8d} KB")
print(f"zip: {Path('/kaggle/working/outputs.zip').stat().st_size // 1024} KB")